### Cell 03.01 — load the frozen framework-map checkpoint

In [ ]:
# Cell 03.01
# Load the frozen de novo linkage-map checkpoint.
#
# IMPORTANT:
# This notebook does NOT use fxh_maps.xlsx.

from pathlib import Path
import pandas as pd
import numpy as np


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()


map_file = (
    PROJECT_ROOT
    / "results"
    / "linkage_map"
    / "flyer_hartwig_provisional_framework_map.xlsx"
)


assert map_file.exists(), (
    f"Map checkpoint not found:\n{map_file}"
)


framework_map = pd.read_excel(
    map_file,
    sheet_name="ordered_framework"
)

all_group_associated = pd.read_excel(
    map_file,
    sheet_name="all_group_associated"
)


print("FROZEN DE NOVO MAP LOADED")
print("=" * 100)

print(
    "Ordered framework markers:",
    framework_map["marker"].nunique()
)

print(
    "Working linkage groups:",
    framework_map["working_group"].nunique()
)

print(
    "All group-associated markers:",
    all_group_associated["marker"].nunique()
)

print("\nNo historical FxH map is being used.")

### Cell 03.02 — identify directly anchorable classic SSR markers
* For the first anchoring pass, we should use exact marker-name matches only.

In [ ]:
# Cell 03.02
# Identify classic SSR markers likely represented in the
# modern SoyBase SoySSR collection.

import re


def classify_marker(marker):

    marker = str(marker)

    if re.match(r"^Satt", marker, flags=re.IGNORECASE):
        return "Satt"

    if re.match(r"^Sat_", marker, flags=re.IGNORECASE):
        return "Sat_"

    return "other"


framework_map[
    "marker_family"
] = (
    framework_map[
        "marker"
    ].apply(
        classify_marker
    )
)


framework_ssr = (
    framework_map.loc[
        framework_map[
            "marker_family"
        ].isin(
            ["Satt", "Sat_"]
        )
    ]
    .copy()
)


print("CLASSIC SSR ANCHOR CANDIDATES")
print("=" * 100)

print(
    framework_map[
        "marker_family"
    ].value_counts()
)

print(
    "\nFramework SSR candidates:",
    len(framework_ssr)
)

print(
    "Non-SSR framework markers:",
    (
        framework_map[
            "marker_family"
        ] == "other"
    ).sum()
)


display(
    framework_ssr[
        [
            "working_group",
            "order_position",
            "marker",
            "kosambi_cm_provisional",
            "marker_family"
        ]
    ].head(40)
)

### Cell 03.03 — download the current SoyBase Wm82.gnm6 SoySSR file
* The SoyBase DataStore currently lists this exact GFF3 file for Wm82.gnm6.

In [ ]:
# Cell 03.03
# Locate the SoyBase Wm82.gnm6 SoySSR reference locally.
#
# Direct automated retrieval from the SoyBase file endpoint
# currently returns HTTP 403, so download the file once in
# your browser and place it in the project's reference folder.

from pathlib import Path
import shutil


reference_dir = (
    PROJECT_ROOT
    / "reference"
    / "soybase"
)

reference_dir.mkdir(
    parents=True,
    exist_ok=True
)


soyssr_filename = (
    "glyma.Wm82.gnm6.mrk.SoySSR.gff3.gz"
)

soyssr_gff = (
    reference_dir
    / soyssr_filename
)


# Check the usual macOS Downloads directory too.
downloads_candidate = (
    Path.home()
    / "Downloads"
    / soyssr_filename
)


if soyssr_gff.exists():

    print("Reference already present in project.")

elif downloads_candidate.exists():

    shutil.copy2(
        downloads_candidate,
        soyssr_gff
    )

    print(
        "Copied reference from Downloads "
        "into project."
    )

else:

    print(
        "REFERENCE NOT FOUND\n"
        "\n"
        "Please download this file manually "
        "with your browser:\n"
        f"{soyssr_filename}\n"
        "\n"
        "Then place it either in:\n"
        f"{reference_dir}\n"
        "\n"
        "or leave it in ~/Downloads and "
        "rerun this cell."
    )


if soyssr_gff.exists():

    print("\nREFERENCE READY")
    print("=" * 100)

    print(soyssr_gff)

    print(
        "\nFile size:",
        round(
            soyssr_gff.stat().st_size / 1024,
            2
        ),
        "KB"
    )

### Cell 03.04 — inspect and parse the SoySSR GFF3
* We should inspect the actual GFF attributes before assuming where the marker name lives.

In [ ]:
# Cell 03.04
# Read and inspect the Wm82.gnm6 SoySSR GFF3.

import gzip


gff_rows = []


with gzip.open(
    soyssr_gff,
    "rt"
) as handle:

    for line in handle:

        if line.startswith("#"):
            continue

        fields = (
            line.rstrip("\n")
            .split("\t")
        )

        if len(fields) != 9:
            continue


        gff_rows.append({
            "seqid": fields[0],
            "source": fields[1],
            "type": fields[2],
            "start": int(fields[3]),
            "end": int(fields[4]),
            "score": fields[5],
            "strand": fields[6],
            "phase": fields[7],
            "attributes": fields[8]
        })


soyssr_gff_df = pd.DataFrame(
    gff_rows
)


print("SOYBASE Wm82.gnm6 SoySSR GFF3")
print("=" * 110)

print(
    "Features:",
    len(soyssr_gff_df)
)

print(
    "\nSequence IDs:"
)

print(
    soyssr_gff_df[
        "seqid"
    ].value_counts()
)


print("\nFirst 20 records")
print("-" * 110)

display(
    soyssr_gff_df.head(20)
)


print("\nFirst 20 attribute strings")
print("-" * 110)

for value in (
    soyssr_gff_df[
        "attributes"
    ]
    .head(20)
):

    print(value)

### Cell 03.05 — parse SoyBase marker names and chromosomes

In [ ]:
# Cell 03.05
# Parse marker metadata from the SoyBase Wm82.gnm6 SoySSR GFF3.

def parse_gff_attributes(attr_string):

    result = {}

    for item in str(attr_string).split(";"):

        if "=" in item:
            key, value = item.split("=", 1)
            result[key] = value

    return result


soyssr_parsed = soyssr_gff_df.copy()


parsed_attrs = (
    soyssr_parsed["attributes"]
    .apply(parse_gff_attributes)
)


soyssr_parsed["marker_name"] = (
    parsed_attrs.apply(
        lambda x: x.get("Name")
    )
)

soyssr_parsed["marker_id"] = (
    parsed_attrs.apply(
        lambda x: x.get("ID")
    )
)

soyssr_parsed["alias"] = (
    parsed_attrs.apply(
        lambda x: x.get("alias")
    )
)

soyssr_parsed["motif"] = (
    parsed_attrs.apply(
        lambda x: x.get("motif")
    )
)


# Extract modern chromosome label.
soyssr_parsed["physical_chromosome"] = (
    soyssr_parsed["seqid"]
    .str.extract(
        r"(Gm\d{2})$",
        expand=False
    )
)


# Use midpoint as the physical marker coordinate.
soyssr_parsed["physical_bp"] = (
    (
        soyssr_parsed["start"]
        +
        soyssr_parsed["end"]
    )
    / 2
).round().astype("Int64")


print("PARSED SOYBASE SSR REFERENCE")
print("=" * 110)

print(
    "Reference markers:",
    len(soyssr_parsed)
)

print(
    "Unique marker names:",
    soyssr_parsed[
        "marker_name"
    ].nunique()
)

print(
    "Chromosomes:",
    soyssr_parsed[
        "physical_chromosome"
    ].nunique()
)

print(
    "\nDuplicate marker names:",
    soyssr_parsed[
        "marker_name"
    ].duplicated(
        keep=False
    ).sum()
)


display(
    soyssr_parsed[
        [
            "marker_name",
            "physical_chromosome",
            "start",
            "end",
            "physical_bp",
            "alias",
            "motif"
        ]
    ].head(20)
)

### Cell 03.06 — exact-match the FxH framework SSRs to Wm82.gnm6
* No suffix stripping yet. Satt357a stays Satt357a, not Satt357.

In [ ]:
# Cell 03.06
# Exact-name physical anchoring of framework SSR markers.

soyssr_lookup = (
    soyssr_parsed[
        [
            "marker_name",
            "physical_chromosome",
            "start",
            "end",
            "physical_bp",
            "alias",
            "motif"
        ]
    ]
    .copy()
)


framework_ssr_anchors = (
    framework_ssr
    .merge(
        soyssr_lookup,
        left_on="marker",
        right_on="marker_name",
        how="left",
        validate="many_to_one"
    )
)


framework_ssr_anchors[
    "exact_match"
] = (
    framework_ssr_anchors[
        "physical_chromosome"
    ].notna()
)


print("EXACT SSR PHYSICAL ANCHORING")
print("=" * 110)

print(
    "Framework SSR candidates:",
    len(framework_ssr_anchors)
)

print(
    "Exact Wm82.gnm6 matches:",
    framework_ssr_anchors[
        "exact_match"
    ].sum()
)

print(
    "Unmatched:",
    (
        ~framework_ssr_anchors[
            "exact_match"
        ]
    ).sum()
)

print(
    "Exact-match rate:",
    round(
        100
        *
        framework_ssr_anchors[
            "exact_match"
        ].mean(),
        2
    ),
    "%"
)


print("\nMATCHED EXAMPLES")
print("-" * 110)

display(
    framework_ssr_anchors.loc[
        framework_ssr_anchors[
            "exact_match"
        ],
        [
            "working_group",
            "order_position",
            "marker",
            "kosambi_cm_provisional",
            "physical_chromosome",
            "physical_bp"
        ]
    ].head(40)
)


print("\nUNMATCHED SSR NAMES")
print("-" * 110)

display(
    framework_ssr_anchors.loc[
        ~framework_ssr_anchors[
            "exact_match"
        ],
        [
            "working_group",
            "order_position",
            "marker",
            "marker_family"
        ]
    ]
    .sort_values(
        [
            "working_group",
            "order_position"
        ]
    )
)

### Cell 03.07 — chromosome votes for each de novo linkage group
* This is the important one.

In [ ]:
# Cell 03.07
# Summarize independent physical chromosome support for each pLG.

matched_anchors = (
    framework_ssr_anchors.loc[
        framework_ssr_anchors[
            "exact_match"
        ]
    ]
    .copy()
)


chromosome_votes = (
    matched_anchors
    .groupby(
        [
            "working_group",
            "physical_chromosome"
        ]
    )
    .size()
    .reset_index(
        name="n_anchors"
    )
)


group_anchor_totals = (
    matched_anchors
    .groupby(
        "working_group"
    )
    .size()
    .reset_index(
        name="n_total_exact_anchors"
    )
)


chromosome_votes = (
    chromosome_votes
    .merge(
        group_anchor_totals,
        on="working_group",
        how="left"
    )
)


chromosome_votes[
    "anchor_fraction"
] = (
    chromosome_votes[
        "n_anchors"
    ]
    /
    chromosome_votes[
        "n_total_exact_anchors"
    ]
)


chromosome_votes = (
    chromosome_votes
    .sort_values(
        [
            "working_group",
            "n_anchors",
            "physical_chromosome"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
)


print("PHYSICAL CHROMOSOME VOTES BY DE NOVO LINKAGE GROUP")
print("=" * 120)

display(
    chromosome_votes
)


# Identify dominant chromosome per pLG.
dominant_chromosome = (
    chromosome_votes
    .groupby(
        "working_group",
        group_keys=False
    )
    .head(1)
    .copy()
)


def anchor_confidence(row):

    n = row["n_anchors"]
    frac = row["anchor_fraction"]

    if n >= 5 and frac >= 0.80:
        return "high"

    elif n >= 3 and frac >= 0.70:
        return "moderate"

    elif n >= 2 and frac >= 0.60:
        return "tentative"

    else:
        return "insufficient"


dominant_chromosome[
    "physical_assignment_confidence"
] = (
    dominant_chromosome.apply(
        anchor_confidence,
        axis=1
    )
)


print("\nDOMINANT PHYSICAL CHROMOSOME ASSIGNMENT")
print("=" * 120)

display(
    dominant_chromosome[
        [
            "working_group",
            "physical_chromosome",
            "n_anchors",
            "n_total_exact_anchors",
            "anchor_fraction",
            "physical_assignment_confidence"
        ]
    ]
)

### Cell 03.08 — check physical-order concordance within each pLG
* Once several markers from a pLG land on the same physical chromosome, the physical coordinates should usually progress approximately monotonically with genetic order. The whole pLG may be reversed, which is completely fine.

In [ ]:
# Cell 03.08
# Test genetic-order versus physical-order concordance
# within the dominant chromosome of each pLG.

from scipy.stats import spearmanr


order_concordance_records = []


for _, group_row in dominant_chromosome.iterrows():

    group = group_row["working_group"]
    chrom = group_row["physical_chromosome"]


    temp = (
        matched_anchors.loc[
            (
                matched_anchors[
                    "working_group"
                ] == group
            )
            &
            (
                matched_anchors[
                    "physical_chromosome"
                ] == chrom
            )
        ]
        .sort_values(
            "order_position"
        )
        .copy()
    )


    n = len(temp)


    if n >= 3:

        rho, p_value = spearmanr(
            temp[
                "order_position"
            ],
            temp[
                "physical_bp"
            ]
        )

    else:

        rho = np.nan
        p_value = np.nan


    if pd.isna(rho):

        orientation = (
            "insufficient_anchors"
        )

    elif rho >= 0:

        orientation = (
            "forward"
        )

    else:

        orientation = (
            "reverse"
        )


    order_concordance_records.append({
        "working_group":
            group,

        "physical_chromosome":
            chrom,

        "n_concordant_anchors":
            n,

        "spearman_rho":
            rho,

        "spearman_p":
            p_value,

        "orientation":
            orientation,

        "abs_spearman_rho":
            abs(rho)
            if not pd.isna(rho)
            else np.nan
    })


order_concordance = pd.DataFrame(
    order_concordance_records
)


print("GENETIC-vs-PHYSICAL ORDER CONCORDANCE")
print("=" * 120)

display(
    order_concordance
    .sort_values(
        [
            "n_concordant_anchors",
            "abs_spearman_rho"
        ],
        ascending=[
            False,
            False
        ]
    )
)

### Cell 03.09 — show exact anchors in genetic order and identify chromosome blocks

In [ ]:
# Cell 03.09
# Examine physical chromosome assignments along each genetic linkage group.

anchor_order_table = (
    matched_anchors[
        [
            "working_group",
            "order_position",
            "marker",
            "kosambi_cm_provisional",
            "physical_chromosome",
            "physical_bp"
        ]
    ]
    .sort_values(
        [
            "working_group",
            "order_position"
        ]
    )
    .copy()
)


print("EXACT PHYSICAL ANCHORS ALONG EACH DE NOVO LINKAGE GROUP")
print("=" * 120)


for group, temp in anchor_order_table.groupby(
    "working_group",
    sort=True
):

    print(f"\n{group}")
    print("-" * 120)

    display(
        temp.reset_index(
            drop=True
        )
    )

### Cell 03.10 — classify chromosome consistency without forcing assignments

In [ ]:
# Cell 03.10
# Conservative chromosome-consistency classification for each pLG.

group_diagnostics = []


for group in sorted(
    framework_map["working_group"].unique()
):

    temp = matched_anchors.loc[
        matched_anchors[
            "working_group"
        ] == group
    ].copy()


    n_total = len(temp)

    if n_total == 0:

        group_diagnostics.append({
            "working_group":
                group,

            "n_exact_anchors":
                0,

            "dominant_chromosome":
                None,

            "dominant_n":
                0,

            "dominant_fraction":
                np.nan,

            "n_physical_chromosomes":
                0,

            "second_chromosome":
                None,

            "second_n":
                0,

            "chromosome_status":
                "unresolved_no_exact_anchors"
        })

        continue


    counts = (
        temp[
            "physical_chromosome"
        ]
        .value_counts()
    )


    dominant_chr = counts.index[0]
    dominant_n = int(counts.iloc[0])

    dominant_fraction = (
        dominant_n
        /
        n_total
    )


    if len(counts) >= 2:

        second_chr = counts.index[1]
        second_n = int(
            counts.iloc[1]
        )

    else:

        second_chr = None
        second_n = 0


    # Conservative evidence categories.
    if (
        dominant_n >= 5
        and
        dominant_fraction >= 0.80
    ):

        status = (
            "strong_single_chromosome"
        )

    elif (
        dominant_n >= 3
        and
        dominant_fraction >= 0.70
    ):

        status = (
            "supported_single_chromosome"
        )

    elif (
        dominant_n >= 3
        and
        second_n >= 2
    ):

        status = (
            "possible_chimeric_or_merged"
        )

    elif (
        dominant_n >= 2
        and
        dominant_fraction >= 0.60
    ):

        status = (
            "tentative_single_chromosome"
        )

    elif n_total >= 3:

        status = (
            "mixed_unresolved"
        )

    else:

        status = (
            "insufficient_anchor_count"
        )


    group_diagnostics.append({
        "working_group":
            group,

        "n_exact_anchors":
            n_total,

        "dominant_chromosome":
            dominant_chr,

        "dominant_n":
            dominant_n,

        "dominant_fraction":
            dominant_fraction,

        "n_physical_chromosomes":
            len(counts),

        "second_chromosome":
            second_chr,

        "second_n":
            second_n,

        "chromosome_status":
            status
    })


group_diagnostics = pd.DataFrame(
    group_diagnostics
)


print("CONSERVATIVE PHYSICAL-ANCHOR DIAGNOSTICS")
print("=" * 120)

display(
    group_diagnostics
)

### Cell 03.11 — identify chromosomes represented by multiple pLGs
* This tells us whether some genotype-derived linkage groups are probably fragments of the same physical chromosome.

In [ ]:
# Cell 03.11
# Identify physical chromosomes supported by more than one pLG.

usable_assignments = (
    group_diagnostics.loc[
        group_diagnostics[
            "dominant_n"
        ] >= 2
    ]
    .copy()
)


chromosome_to_plg = (
    usable_assignments
    .groupby(
        "dominant_chromosome"
    )
    .agg(
        n_plgs=(
            "working_group",
            "nunique"
        ),

        linkage_groups=(
            "working_group",
            lambda x:
                ", ".join(
                    sorted(x)
                )
        ),

        total_dominant_anchors=(
            "dominant_n",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        [
            "n_plgs",
            "dominant_chromosome"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print("PHYSICAL CHROMOSOMES REPRESENTED BY MULTIPLE pLGs")
print("=" * 110)

display(
    chromosome_to_plg
)


print("\nPOTENTIAL SPLIT-CHROMOSOME CASES")
print("-" * 110)

display(
    chromosome_to_plg.loc[
        chromosome_to_plg[
            "n_plgs"
        ] > 1
    ]
)

### Cell 03.12 — cautiously investigate the 50 unmatched SSR names
* A large proportion of the unmatched names contain suffixes such as Satt357a, Satt079b, Satt163a, Satt163b, Satt163c, Satt309a, and Satt309b
* We should see whether the unsuffixed assay name exists in SoyBase, but we will not treat that as an exact physical anchor.

In [ ]:
# Cell 03.12
# Find possible parent-assay matches for suffix-bearing SSR names.
#
# IMPORTANT:
# These are candidate assay-family mappings only.
# They are NOT promoted to exact physical anchors.

unmatched_ssr = (
    framework_ssr_anchors.loc[
        ~framework_ssr_anchors[
            "exact_match"
        ]
    ]
    .copy()
)


unmatched_ssr[
    "base_marker_candidate"
] = (
    unmatched_ssr[
        "marker"
    ]
    .str.replace(
        r"([a-z])$",
        "",
        regex=True
    )
)


suffix_candidates = (
    unmatched_ssr
    .merge(
        soyssr_lookup.rename(
            columns={
                "marker_name":
                    "base_marker_candidate",

                "physical_chromosome":
                    "candidate_chromosome",

                "physical_bp":
                    "candidate_bp"
            }
        )[
            [
                "base_marker_candidate",
                "candidate_chromosome",
                "candidate_bp"
            ]
        ],
        on="base_marker_candidate",
        how="left"
    )
)


suffix_candidates[
    "base_name_found"
] = (
    suffix_candidates[
        "candidate_chromosome"
    ].notna()
)


suffix_candidates[
    "anchor_status"
] = np.where(
    suffix_candidates[
        "base_name_found"
    ],
    "assay_family_candidate_only",
    "no_reference_match"
)


print("UNMATCHED SSR ASSAY-FAMILY SEARCH")
print("=" * 120)

print(
    "Unmatched exact SSRs:",
    len(suffix_candidates)
)

print(
    "Unsuffixed assay names found:",
    suffix_candidates[
        "base_name_found"
    ].sum()
)


display(
    suffix_candidates[
        [
            "working_group",
            "order_position",
            "marker",
            "base_marker_candidate",
            "candidate_chromosome",
            "candidate_bp",
            "anchor_status"
        ]
    ]
    .sort_values(
        [
            "working_group",
            "order_position"
        ]
    )
)

### Cell 03.13 — build a unified two-tier physical-anchor table

In [ ]:
# Cell 03.13
# Combine exact SSR anchors with suffix-derived assay-family evidence.
#
# Exact name matches remain PRIMARY evidence.
# Suffix-derived base-assay matches remain SECONDARY evidence.

# ------------------------------------------------------------
# Exact anchors
# ------------------------------------------------------------

exact_evidence = (
    framework_ssr_anchors.loc[
        framework_ssr_anchors["exact_match"]
    ]
    [
        [
            "working_group",
            "order_position",
            "marker",
            "kosambi_cm_provisional",
            "physical_chromosome",
            "physical_bp"
        ]
    ]
    .copy()
)

exact_evidence["base_assay"] = (
    exact_evidence["marker"]
)

exact_evidence["evidence_class"] = (
    "exact"
)


# ------------------------------------------------------------
# Assay-family-only candidates
# ------------------------------------------------------------

family_evidence = (
    suffix_candidates.loc[
        suffix_candidates[
            "base_name_found"
        ]
    ]
    [
        [
            "working_group",
            "order_position",
            "marker",
            "kosambi_cm_provisional",
            "base_marker_candidate",
            "candidate_chromosome",
            "candidate_bp"
        ]
    ]
    .copy()
)

family_evidence = (
    family_evidence.rename(
        columns={
            "base_marker_candidate":
                "base_assay",

            "candidate_chromosome":
                "physical_chromosome",

            "candidate_bp":
                "physical_bp"
        }
    )
)

family_evidence["evidence_class"] = (
    "assay_family"
)


# ------------------------------------------------------------
# Harmonize columns
# ------------------------------------------------------------

combined_anchor_evidence = pd.concat(
    [
        exact_evidence,
        family_evidence[
            exact_evidence.columns
        ]
    ],
    ignore_index=True
)


combined_anchor_evidence = (
    combined_anchor_evidence
    .sort_values(
        [
            "working_group",
            "order_position",
            "evidence_class"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("COMBINED PHYSICAL-ANCHOR EVIDENCE")
print("=" * 120)

print(
    combined_anchor_evidence[
        "evidence_class"
    ].value_counts()
)

display(
    combined_anchor_evidence.head(50)
)

### Cell 03.14 — collapse suffix bands to independent SSR assays
* This is important. Satt574a and Satt574b, for example, must not count as two independent physical anchors.

In [ ]:
# Cell 03.14
# Collapse multiple segregating bands derived from the same SSR assay.
#
# One base assay = one physical-reference observation
# within a linkage group.

independent_assay_evidence = (
    combined_anchor_evidence
    .sort_values(
        [
            "working_group",
            "base_assay",
            "evidence_class"
        ]
    )
    .drop_duplicates(
        subset=[
            "working_group",
            "base_assay",
            "physical_chromosome"
        ],
        keep="first"
    )
    .copy()
)


print("INDEPENDENT SSR ASSAY EVIDENCE")
print("=" * 120)

print(
    "Raw evidence rows:",
    len(combined_anchor_evidence)
)

print(
    "Independent group-assay-chromosome observations:",
    len(independent_assay_evidence)
)


assay_vote_summary = (
    independent_assay_evidence
    .groupby(
        [
            "working_group",
            "physical_chromosome"
        ]
    )
    .agg(
        n_independent_assays=(
            "base_assay",
            "nunique"
        ),

        n_exact_assays=(
            "evidence_class",
            lambda x:
                (x == "exact").sum()
        ),

        n_family_assays=(
            "evidence_class",
            lambda x:
                (x == "assay_family").sum()
        )
    )
    .reset_index()
    .sort_values(
        [
            "working_group",
            "n_independent_assays"
        ],
        ascending=[
            True,
            False
        ]
    )
)


display(
    assay_vote_summary
)

### Cell 03.15 — generate a conservative updated chromosome-assignment table

In [ ]:
# Cell 03.15
# Updated chromosome assignments using:
#   1. exact anchors as primary evidence
#   2. independent assay-family evidence as secondary support
#
# No one-to-one chromosome assignment is forced.

updated_assignments = []


for group in sorted(
    framework_map[
        "working_group"
    ].unique()
):

    temp = assay_vote_summary.loc[
        assay_vote_summary[
            "working_group"
        ] == group
    ].copy()


    if temp.empty:

        updated_assignments.append({
            "working_group":
                group,

            "candidate_chromosome":
                None,

            "independent_assays":
                0,

            "exact_assays":
                0,

            "family_assays":
                0,

            "second_chromosome":
                None,

            "second_assays":
                0,

            "assignment_status":
                "unresolved"
        })

        continue


    temp = (
        temp.sort_values(
            [
                "n_independent_assays",
                "n_exact_assays"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


    top = temp.iloc[0]


    if len(temp) > 1:

        second = temp.iloc[1]

        second_chr = (
            second[
                "physical_chromosome"
            ]
        )

        second_n = int(
            second[
                "n_independent_assays"
            ]
        )

    else:

        second_chr = None
        second_n = 0


    top_n = int(
        top[
            "n_independent_assays"
        ]
    )

    exact_n = int(
        top[
            "n_exact_assays"
        ]
    )

    family_n = int(
        top[
            "n_family_assays"
        ]
    )


    # Conservative rules.
    if (
        exact_n >= 5
        and
        second_n <= 1
    ):

        status = (
            "strong"
        )

    elif (
        exact_n >= 3
        and
        top_n >= 3
        and
        second_n <= 1
    ):

        status = (
            "supported"
        )

    elif (
        exact_n >= 2
        and
        top_n >= 2
        and
        second_n <= 1
    ):

        status = (
            "tentative"
        )

    elif (
        exact_n >= 1
        and
        top_n >= 3
        and
        second_n <= 1
    ):

        status = (
            "supported_by_exact_plus_family"
        )

    elif (
        exact_n == 0
        and
        top_n >= 2
        and
        second_n == 0
    ):

        status = (
            "tentative_family_only"
        )

    elif second_n >= 2:

        status = (
            "mixed_or_chimeric"
        )

    else:

        status = (
            "insufficient"
        )


    updated_assignments.append({
        "working_group":
            group,

        "candidate_chromosome":
            top[
                "physical_chromosome"
            ],

        "independent_assays":
            top_n,

        "exact_assays":
            exact_n,

        "family_assays":
            family_n,

        "second_chromosome":
            second_chr,

        "second_assays":
            second_n,

        "assignment_status":
            status
    })


updated_assignments = pd.DataFrame(
    updated_assignments
)


print("UPDATED INDEPENDENT PHYSICAL ASSIGNMENTS")
print("=" * 120)

display(
    updated_assignments
)

### Cell 03.16 — explicitly find chromosome transitions along pLG01 and pLG02

In [ ]:
# Cell 03.16
# Inspect physical-anchor chromosome transitions
# in the two most structurally suspicious linkage groups.

for group in [
    "pLG01",
    "pLG02"
]:

    temp = (
        combined_anchor_evidence.loc[
            combined_anchor_evidence[
                "working_group"
            ] == group
        ]
        .sort_values(
            "order_position"
        )
        .copy()
    )


    temp[
        "previous_chromosome"
    ] = (
        temp[
            "physical_chromosome"
        ]
        .shift(1)
    )


    temp[
        "chromosome_change"
    ] = (
        temp[
            "physical_chromosome"
        ]
        !=
        temp[
            "previous_chromosome"
        ]
    )


    print(
        f"\n{group} — PHYSICAL ANCHOR SEQUENCE"
    )

    print(
        "=" * 120
    )


    display(
        temp[
            [
                "order_position",
                "marker",
                "base_assay",
                "evidence_class",
                "kosambi_cm_provisional",
                "physical_chromosome",
                "physical_bp",
                "chromosome_change"
            ]
        ]
    )

### Cell 03.17 — recover the ordered marker lists for pLG01 and pLG02

In [ ]:
# Cell 03.17
# Show every ordered framework marker around the two mixed groups,
# including markers without physical anchors.

suspect_groups = ["pLG01", "pLG02"]

suspect_framework = (
    framework_map.loc[
        framework_map["working_group"].isin(
            suspect_groups
        )
    ]
    [
        [
            "working_group",
            "order_position",
            "marker",
            "kosambi_cm_provisional"
        ]
    ]
    .sort_values(
        [
            "working_group",
            "order_position"
        ]
    )
    .copy()
)


# Add physical evidence where available.
suspect_framework = (
    suspect_framework
    .merge(
        combined_anchor_evidence[
            [
                "working_group",
                "order_position",
                "marker",
                "physical_chromosome",
                "physical_bp",
                "evidence_class"
            ]
        ],
        on=[
            "working_group",
            "order_position",
            "marker"
        ],
        how="left"
    )
)


for group in suspect_groups:

    print(f"\n{group} — COMPLETE ORDERED FRAMEWORK")
    print("=" * 120)

    display(
        suspect_framework.loc[
            suspect_framework[
                "working_group"
            ] == group
        ].reset_index(drop=True)
    )

### Cell 03.18 — identify candidate breakpoint intervals

In [ ]:
# Cell 03.18
# Define focused candidate breakpoint regions for genotype testing.

candidate_breakpoints = pd.DataFrame(
    [
        {
            "working_group": "pLG01",
            "left_order": 20,
            "right_order": 23,
            "reason":
                "transition from dominant Gm06 block to terminal Gm20 block"
        },
        {
            "working_group": "pLG02",
            "left_order": 20,
            "right_order": 22,
            "reason":
                "entry into Gm18-enriched region"
        }
    ]
)


print("CANDIDATE STRUCTURAL BREAKPOINTS")
print("=" * 120)

display(candidate_breakpoints)

### Cell 03.19 — inspect the markers immediately spanning each breakpoint
* This assumes your frozen interval table is still available as provisional_intervals or something similar. To avoid relying on memory of its variable name, we reload it directly from the checkpoint workbook.

In [ ]:
# Cell 03.19
# Reload the frozen interval table and inspect intervals
# around the suspected chromosome transitions.

intervals_frozen = pd.read_excel(
    map_file,
    sheet_name="intervals"
)


print("INTERVAL TABLE COLUMNS")
print("=" * 120)

print(
    intervals_frozen.columns.tolist()
)


for group in suspect_groups:

    print(
        f"\n{group} — INTERVALS"
    )
    print("-" * 120)

    display(
        intervals_frozen.loc[
            intervals_frozen[
                "working_group"
            ] == group
        ]
    )

### Cell 03.20 — summarize our chromosome assignments without renaming groups
* We can already save a provisional anchoring table for provenance.

In [ ]:
# Cell 03.20
# Save current physical-chromosome interpretation.
#
# This does NOT rename linkage groups and does NOT alter the map.

physical_assignment_checkpoint = (
    updated_assignments.copy()
)


physical_assignment_checkpoint[
    "interpretation_note"
] = ""


physical_assignment_checkpoint.loc[
    physical_assignment_checkpoint[
        "working_group"
    ] == "pLG01",
    "interpretation_note"
] = (
    "mixed; dominant Gm06 with terminal exact Gm20 block; "
    "requires genotype breakpoint test"
)


physical_assignment_checkpoint.loc[
    physical_assignment_checkpoint[
        "working_group"
    ] == "pLG02",
    "interpretation_note"
] = (
    "mixed; Gm18 enriched in latter region; "
    "requires genotype breakpoint test"
)


out_file = (
    PROJECT_ROOT
    / "results"
    / "linkage_map"
    / "flyer_hartwig_independent_physical_anchoring_checkpoint.xlsx"
)


with pd.ExcelWriter(
    out_file,
    engine="openpyxl"
) as writer:

    physical_assignment_checkpoint.to_excel(
        writer,
        sheet_name="plg_assignments",
        index=False
    )

    independent_assay_evidence.to_excel(
        writer,
        sheet_name="independent_ssr_evidence",
        index=False
    )

    combined_anchor_evidence.to_excel(
        writer,
        sheet_name="all_anchor_evidence",
        index=False
    )

    anchor_order_table.to_excel(
        writer,
        sheet_name="exact_anchor_order",
        index=False
    )


print("PHYSICAL ANCHORING CHECKPOINT SAVED")
print("=" * 120)
print(out_file)

### Cell 03.21 — reload the 92-RIL genotype matrix

In [ ]:
# Cell 03.21
# Reload genotype data and reproduce the conservative 92-RIL subset
# used for linkage reconstruction.

genotype_file = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "fxh_genotypes.xlsx"
)

assert genotype_file.exists(), (
    f"Genotype file not found:\n{genotype_file}"
)


geno_raw = pd.read_excel(
    genotype_file,
    sheet_name="fxh_genotypes"
)


geno_raw["ril"] = (
    geno_raw["ril"]
    .astype(str)
    .str.strip()
)


excluded_rils = {
    "fxh_ril_03",
    "fxh_ril_43"
}


geno_92 = (
    geno_raw.loc[
        ~geno_raw["ril"].isin(
            excluded_rils
        )
    ]
    .copy()
)


marker_columns = [
    c for c in geno_92.columns
    if c != "ril"
]


geno_92[
    marker_columns
] = (
    geno_92[
        marker_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


print("GENOTYPE MATRIX FOR BREAKPOINT TESTING")
print("=" * 110)

print(
    "RILs:",
    len(geno_92)
)

print(
    "Markers:",
    len(marker_columns)
)

print(
    "Excluded:",
    sorted(excluded_rils)
)

### Cell 03.22 — define the same phase-independent linkage statistics
* This uses the same pairwise logic as the reconstructed map.

In [ ]:
# Cell 03.22
# Phase-independent pairwise linkage statistics for 0/2 RIL genotypes.

import math


def pairwise_linkage_stats(
    genotype_df,
    marker_a,
    marker_b
):

    a = genotype_df[
        marker_a
    ]

    b = genotype_df[
        marker_b
    ]


    valid = (
        a.isin([0, 2])
        &
        b.isin([0, 2])
    )


    a = a.loc[valid]
    b = b.loc[valid]


    n = len(a)


    if n == 0:

        return {
            "n_overlap": 0,
            "R_ril": np.nan,
            "lod": np.nan
        }


    discordant = int(
        (a != b).sum()
    )


    # Phase-independent:
    # choose whichever phase gives R <= 0.5.
    recombinants = min(
        discordant,
        n - discordant
    )


    R = (
        recombinants
        /
        n
    )


    if R == 0:

        lod = (
            n
            *
            math.log10(2)
        )

    elif R == 0.5:

        lod = 0.0

    else:

        lod = (
            (n - recombinants)
            *
            math.log10(
                1 - R
            )
            +
            recombinants
            *
            math.log10(
                R
            )
            -
            n
            *
            math.log10(
                0.5
            )
        )


    return {
        "n_overlap":
            n,

        "R_ril":
            R,

        "lod":
            lod
    }


# Sanity check against one frozen pLG01 interval.
test = pairwise_linkage_stats(
    geno_92,
    "PUBTONC2",
    "Satt367"
)


print("SANITY CHECK: PUBTONC2 vs Satt367")
print("=" * 110)

print(test)

print(
    "\nFrozen expected approximately:"
)

print(
    "n_overlap = 76, "
    "R_ril = 0.25, "
    "LOD = 4.3176"
)

### Cell 03.23 — test the pLG01 Gm06 block against the Gm20 block
* Rather than relying on the single PUBTONC2–Satt367 interval, this tests all pairwise links between the two chromosome-supported blocks.

In [ ]:
# Cell 03.23
# Test all pairwise linkage relationships between:
#   pLG01 dominant Gm06 block
#   pLG01 terminal Gm20 block

plg01_gm06 = (
    combined_anchor_evidence.loc[
        (
            combined_anchor_evidence[
                "working_group"
            ] == "pLG01"
        )
        &
        (
            combined_anchor_evidence[
                "physical_chromosome"
            ] == "Gm06"
        ),
        "marker"
    ]
    .drop_duplicates()
    .tolist()
)


plg01_gm20 = (
    combined_anchor_evidence.loc[
        (
            combined_anchor_evidence[
                "working_group"
            ] == "pLG01"
        )
        &
        (
            combined_anchor_evidence[
                "physical_chromosome"
            ] == "Gm20"
        ),
        "marker"
    ]
    .drop_duplicates()
    .tolist()
)


cross_rows = []


for marker_a in plg01_gm06:

    for marker_b in plg01_gm20:

        stats = pairwise_linkage_stats(
            geno_92,
            marker_a,
            marker_b
        )

        cross_rows.append({
            "marker_gm06":
                marker_a,

            "marker_gm20":
                marker_b,

            **stats
        })


plg01_crossblock = pd.DataFrame(
    cross_rows
)


print("pLG01 — Gm06 vs Gm20 CROSS-BLOCK LINKAGE")
print("=" * 120)

print(
    "Gm06 markers:",
    plg01_gm06
)

print(
    "\nGm20 markers:",
    plg01_gm20
)


display(
    plg01_crossblock
    .sort_values(
        [
            "R_ril",
            "lod"
        ],
        ascending=[
            True,
            False
        ]
    )
)


print("\nSUMMARY")
print("-" * 120)

print(
    plg01_crossblock[
        [
            "n_overlap",
            "R_ril",
            "lod"
        ]
    ].describe()
)


print(
    "\nPairs meeting core linkage criterion "
    "(n>=40, R<=0.30, LOD>=3.5):",
    (
        (
            plg01_crossblock[
                "n_overlap"
            ] >= 40
        )
        &
        (
            plg01_crossblock[
                "R_ril"
            ] <= 0.30
        )
        &
        (
            plg01_crossblock[
                "lod"
            ] >= 3.5
        )
    ).sum(),
    "of",
    len(plg01_crossblock)
)

### Cell 03.24 — do the analogous test for pLG02
* For pLG02, compare the dominant Gm18-supported markers against all non-Gm18 physically anchored markers.

In [ ]:
# Cell 03.24
# Test whether the Gm18-supported portion of pLG02
# is broadly linked to the conflicting physical anchors.

plg02_evidence = (
    combined_anchor_evidence.loc[
        combined_anchor_evidence[
            "working_group"
        ] == "pLG02"
    ]
    .copy()
)


plg02_gm18 = (
    plg02_evidence.loc[
        plg02_evidence[
            "physical_chromosome"
        ] == "Gm18",
        "marker"
    ]
    .drop_duplicates()
    .tolist()
)


plg02_other = (
    plg02_evidence.loc[
        plg02_evidence[
            "physical_chromosome"
        ] != "Gm18",
        "marker"
    ]
    .drop_duplicates()
    .tolist()
)


cross_rows = []


for marker_a in plg02_gm18:

    for marker_b in plg02_other:

        stats = pairwise_linkage_stats(
            geno_92,
            marker_a,
            marker_b
        )

        cross_rows.append({
            "marker_gm18":
                marker_a,

            "marker_other":
                marker_b,

            "other_physical_chr":
                plg02_evidence.loc[
                    plg02_evidence[
                        "marker"
                    ] == marker_b,
                    "physical_chromosome"
                ].iloc[0],

            **stats
        })


plg02_crossblock = pd.DataFrame(
    cross_rows
)


print("pLG02 — Gm18 vs NON-Gm18 CROSS-BLOCK LINKAGE")
print("=" * 120)


display(
    plg02_crossblock
    .sort_values(
        [
            "R_ril",
            "lod"
        ],
        ascending=[
            True,
            False
        ]
    )
)


print("\nSUMMARY BY CONFLICTING PHYSICAL CHROMOSOME")
print("-" * 120)


plg02_cross_summary = (
    plg02_crossblock
    .groupby(
        "other_physical_chr"
    )
    .agg(
        n_pairs=(
            "R_ril",
            "size"
        ),

        median_R=(
            "R_ril",
            "median"
        ),

        min_R=(
            "R_ril",
            "min"
        ),

        median_LOD=(
            "lod",
            "median"
        ),

        max_LOD=(
            "lod",
            "max"
        ),

        n_core_links=(
            "lod",
            lambda x: np.nan
        )
    )
    .reset_index()
)


# Calculate core-link counts separately.
for chrom in (
    plg02_cross_summary[
        "other_physical_chr"
    ]
):

    temp = (
        plg02_crossblock.loc[
            plg02_crossblock[
                "other_physical_chr"
            ] == chrom
        ]
    )

    n_core = (
        (
            temp[
                "n_overlap"
            ] >= 40
        )
        &
        (
            temp[
                "R_ril"
            ] <= 0.30
        )
        &
        (
            temp[
                "lod"
            ] >= 3.5
        )
    ).sum()

    plg02_cross_summary.loc[
        plg02_cross_summary[
            "other_physical_chr"
        ] == chrom,
        "n_core_links"
    ] = n_core


display(
    plg02_cross_summary
)

### Cell 03.25 — test the pLG01 bridge markers against both chromosome blocks

In [ ]:
# Cell 03.25
# Determine whether PWL1 / PUBTONC2 form a narrow chaining bridge
# between the otherwise unlinked Gm06 and Gm20 blocks.

bridge_markers = [
    "PWL1",
    "PUBTONC2"
]


bridge_rows = []


for bridge in bridge_markers:

    for target_block, target_markers in [
        ("Gm06_block", plg01_gm06),
        ("Gm20_block", plg01_gm20)
    ]:

        for target in target_markers:

            stats = pairwise_linkage_stats(
                geno_92,
                bridge,
                target
            )

            bridge_rows.append({
                "bridge_marker": bridge,
                "target_block": target_block,
                "target_marker": target,
                **stats
            })


plg01_bridge_links = pd.DataFrame(
    bridge_rows
)


plg01_bridge_links[
    "core_link"
] = (
    (plg01_bridge_links["n_overlap"] >= 40)
    &
    (plg01_bridge_links["R_ril"] <= 0.30)
    &
    (plg01_bridge_links["lod"] >= 3.5)
)


print("pLG01 BRIDGE-MARKER LINKAGE")
print("=" * 120)

display(
    plg01_bridge_links
    .sort_values(
        [
            "bridge_marker",
            "target_block",
            "R_ril"
        ]
    )
)


print("\nSUMMARY BY BRIDGE MARKER AND BLOCK")
print("-" * 120)

bridge_summary = (
    plg01_bridge_links
    .groupby(
        [
            "bridge_marker",
            "target_block"
        ]
    )
    .agg(
        n_pairs=("R_ril", "size"),
        median_R=("R_ril", "median"),
        min_R=("R_ril", "min"),
        median_LOD=("lod", "median"),
        max_LOD=("lod", "max"),
        n_core_links=("core_link", "sum")
    )
    .reset_index()
)

display(bridge_summary)

### Cell 03.26 — compare within-block versus between-block linkage in pLG01

In [ ]:
# Cell 03.26
# Quantify internal cohesion of the two pLG01 physical blocks.

from itertools import combinations


def block_pairwise_summary(
    genotype_df,
    markers,
    label
):

    rows = []

    for a, b in combinations(markers, 2):

        stats = pairwise_linkage_stats(
            genotype_df,
            a,
            b
        )

        rows.append({
            "comparison": label,
            "marker_a": a,
            "marker_b": b,
            **stats
        })

    return pd.DataFrame(rows)


gm06_internal = block_pairwise_summary(
    geno_92,
    plg01_gm06,
    "Gm06_within"
)


gm20_internal = block_pairwise_summary(
    geno_92,
    plg01_gm20,
    "Gm20_within"
)


cross_block = (
    plg01_crossblock
    .rename(
        columns={
            "marker_gm06": "marker_a",
            "marker_gm20": "marker_b"
        }
    )
    .copy()
)

cross_block[
    "comparison"
] = "Gm06_vs_Gm20"


plg01_structure_test = pd.concat(
    [
        gm06_internal,
        gm20_internal,
        cross_block[
            gm06_internal.columns
        ]
    ],
    ignore_index=True
)


plg01_structure_test[
    "core_link"
] = (
    (plg01_structure_test["n_overlap"] >= 40)
    &
    (plg01_structure_test["R_ril"] <= 0.30)
    &
    (plg01_structure_test["lod"] >= 3.5)
)


structure_summary = (
    plg01_structure_test
    .groupby("comparison")
    .agg(
        n_pairs=("R_ril", "size"),
        median_R=("R_ril", "median"),
        min_R=("R_ril", "min"),
        max_R=("R_ril", "max"),
        median_LOD=("lod", "median"),
        max_LOD=("lod", "max"),
        n_core_links=("core_link", "sum")
    )
    .reset_index()
)


structure_summary[
    "core_link_fraction"
] = (
    structure_summary[
        "n_core_links"
    ]
    /
    structure_summary[
        "n_pairs"
    ]
)


print("pLG01 BLOCK-COHESION TEST")
print("=" * 120)

display(structure_summary)

### Cell 03.27 — identify which pLG02 physical conflicts are genotype-supported
* For pLG02 we should switch questions. Instead of asking whether to split, identify which conflicting SSR assignments are strongly embedded in the Gm18 genotype cluster.

In [ ]:
# Cell 03.27
# Marker-level diagnosis of pLG02 genotype-versus-reference conflicts.

plg02_marker_conflict = (
    plg02_crossblock
    .copy()
)


plg02_marker_conflict[
    "core_link"
] = (
    (plg02_marker_conflict["n_overlap"] >= 40)
    &
    (plg02_marker_conflict["R_ril"] <= 0.30)
    &
    (plg02_marker_conflict["lod"] >= 3.5)
)


conflict_marker_summary = (
    plg02_marker_conflict
    .groupby(
        [
            "marker_other",
            "other_physical_chr"
        ]
    )
    .agg(
        n_gm18_pairs=(
            "marker_gm18",
            "nunique"
        ),

        median_R_to_gm18=(
            "R_ril",
            "median"
        ),

        min_R_to_gm18=(
            "R_ril",
            "min"
        ),

        median_LOD_to_gm18=(
            "lod",
            "median"
        ),

        max_LOD_to_gm18=(
            "lod",
            "max"
        ),

        n_core_links_to_gm18=(
            "core_link",
            "sum"
        )
    )
    .reset_index()
)


conflict_marker_summary[
    "core_fraction"
] = (
    conflict_marker_summary[
        "n_core_links_to_gm18"
    ]
    /
    conflict_marker_summary[
        "n_gm18_pairs"
    ]
)


def conflict_interpretation(row):

    if (
        row["n_core_links_to_gm18"] >= 3
        and
        row["median_R_to_gm18"] <= 0.20
    ):
        return "strong_genotype_reference_conflict"

    elif row["n_core_links_to_gm18"] >= 1:
        return "partial_genotype_reference_conflict"

    else:
        return "not_embedded_in_gm18_cluster"


conflict_marker_summary[
    "interpretation"
] = (
    conflict_marker_summary.apply(
        conflict_interpretation,
        axis=1
    )
)


print("pLG02 MARKER-LEVEL GENOTYPE–REFERENCE CONFLICTS")
print("=" * 120)

display(
    conflict_marker_summary
    .sort_values(
        [
            "n_core_links_to_gm18",
            "median_R_to_gm18"
        ],
        ascending=[
            False,
            True
        ]
    )
)

### Cell 03.28 — create a formal structural-status table
* This records what the data currently support without actually changing the frozen map.

In [ ]:
# Cell 03.28
# Formal working interpretation of the two physically discordant groups.
#
# This is diagnostic metadata only.
# It does NOT edit the frozen linkage map.

structural_status = pd.DataFrame(
    [
        {
            "working_group": "pLG01",
            "current_interpretation":
                "candidate_false_join_or_chaining",
            "dominant_physical_signal":
                "Gm06 with terminal Gm20 block",
            "genotype_evidence":
                "0 of 39 Gm06-vs-Gm20 pairs meet core linkage criterion",
            "recommended_action":
                "evaluate bridge markers; likely split if within-block cohesion is confirmed"
        },

        {
            "working_group": "pLG02",
            "current_interpretation":
                "genotype_reference_discordance",
            "dominant_physical_signal":
                "Gm18 enriched with multiple conflicting SSR assignments",
            "genotype_evidence":
                "multiple strong cross-label links, including Gm02/Gm12/Gm14 markers",
            "recommended_action":
                "retain genotype group; investigate conflicting marker identities/reference provenance"
        }
    ]
)


print("WORKING STRUCTURAL INTERPRETATION")
print("=" * 120)

display(structural_status)

### Cell 03.29 — test terminal pLG01 markers against Gm06 and Gm20

In [ ]:
# Cell 03.29
# Determine whether the unanchored terminal markers
# belong genetically with the Gm20 fragment.

terminal_unanchored = [
    "OF021100",
    "OH12_300",
    "Satt354"
]


terminal_affinity_rows = []


for marker in terminal_unanchored:

    for block_name, block_markers in [
        ("Gm06_block", plg01_gm06),
        ("Gm20_block", plg01_gm20)
    ]:

        for target in block_markers:

            stats = pairwise_linkage_stats(
                geno_92,
                marker,
                target
            )

            terminal_affinity_rows.append({
                "marker": marker,
                "target_block": block_name,
                "target_marker": target,
                **stats
            })


terminal_affinity = pd.DataFrame(
    terminal_affinity_rows
)


terminal_affinity["core_link"] = (
    (terminal_affinity["n_overlap"] >= 40)
    &
    (terminal_affinity["R_ril"] <= 0.30)
    &
    (terminal_affinity["lod"] >= 3.5)
)


terminal_affinity_summary = (
    terminal_affinity
    .groupby(
        [
            "marker",
            "target_block"
        ]
    )
    .agg(
        n_pairs=("R_ril", "size"),
        median_R=("R_ril", "median"),
        min_R=("R_ril", "min"),
        median_LOD=("lod", "median"),
        max_LOD=("lod", "max"),
        n_core_links=("core_link", "sum")
    )
    .reset_index()
)


terminal_affinity_summary[
    "core_fraction"
] = (
    terminal_affinity_summary[
        "n_core_links"
    ]
    /
    terminal_affinity_summary[
        "n_pairs"
    ]
)


print("pLG01 TERMINAL-MARKER BLOCK AFFINITY")
print("=" * 120)

display(
    terminal_affinity_summary
    .sort_values(
        [
            "marker",
            "target_block"
        ]
    )
)

### Cell 03.30 — construct the proposed pLG01 split without modifying the frozen map

In [ ]:
# Cell 03.30
# Create a proposed structural correction for pLG01.
#
# IMPORTANT:
# This is still a proposal table.
# The frozen original map remains untouched.

plg01_split_proposal = (
    framework_map.loc[
        framework_map[
            "working_group"
        ] == "pLG01"
    ]
    .copy()
)


plg01_split_proposal[
    "proposed_fragment"
] = np.where(
    plg01_split_proposal[
        "order_position"
    ] <= 22,
    "pLG01_Gm06_fragment",
    "pLG01_Gm20_fragment"
)


plg01_split_proposal[
    "proposed_physical_chr"
] = np.where(
    plg01_split_proposal[
        "order_position"
    ] <= 22,
    "Gm06",
    "Gm20"
)


plg01_split_proposal[
    "structural_status"
] = (
    "proposed_after_cross_block_linkage_test"
)


print("PROPOSED pLG01 STRUCTURAL SPLIT")
print("=" * 120)

display(
    plg01_split_proposal[
        [
            "order_position",
            "marker",
            "kosambi_cm_provisional",
            "proposed_fragment",
            "proposed_physical_chr",
            "structural_status"
        ]
    ]
)

### Cell 03.31 — test early versus late pLG02 cohesion

In [ ]:
# Cell 03.31
# Test whether pLG02 consists of two broadly linked blocks
# or whether its continuity depends on chaining.

plg02_ordered = (
    framework_map.loc[
        framework_map[
            "working_group"
        ] == "pLG02"
    ]
    .sort_values(
        "order_position"
    )
    .copy()
)


# Natural diagnostic division:
# early = positions 1-11
# late  = positions 12-33

plg02_early = (
    plg02_ordered.loc[
        plg02_ordered[
            "order_position"
        ] <= 11,
        "marker"
    ]
    .tolist()
)


plg02_late = (
    plg02_ordered.loc[
        plg02_ordered[
            "order_position"
        ] >= 12,
        "marker"
    ]
    .tolist()
)


# Cross-block pairs.
rows = []

for a in plg02_early:

    for b in plg02_late:

        stats = pairwise_linkage_stats(
            geno_92,
            a,
            b
        )

        rows.append({
            "comparison":
                "early_vs_late",

            "marker_a":
                a,

            "marker_b":
                b,

            **stats
        })


plg02_early_late = pd.DataFrame(
    rows
)


# Internal cohesion.
plg02_early_internal = (
    block_pairwise_summary(
        geno_92,
        plg02_early,
        "early_within"
    )
)


plg02_late_internal = (
    block_pairwise_summary(
        geno_92,
        plg02_late,
        "late_within"
    )
)


plg02_structure = pd.concat(
    [
        plg02_early_internal,
        plg02_late_internal,
        plg02_early_late
    ],
    ignore_index=True
)


plg02_structure[
    "core_link"
] = (
    (plg02_structure["n_overlap"] >= 40)
    &
    (plg02_structure["R_ril"] <= 0.30)
    &
    (plg02_structure["lod"] >= 3.5)
)


plg02_structure_summary = (
    plg02_structure
    .groupby(
        "comparison"
    )
    .agg(
        n_pairs=("R_ril", "size"),
        median_R=("R_ril", "median"),
        min_R=("R_ril", "min"),
        max_R=("R_ril", "max"),
        median_LOD=("lod", "median"),
        max_LOD=("lod", "max"),
        n_core_links=("core_link", "sum")
    )
    .reset_index()
)


plg02_structure_summary[
    "core_link_fraction"
] = (
    plg02_structure_summary[
        "n_core_links"
    ]
    /
    plg02_structure_summary[
        "n_pairs"
    ]
)


print("pLG02 EARLY-vs-LATE BLOCK COHESION")
print("=" * 120)

display(
    plg02_structure_summary
)

### Cell 03.32 — investigate potentially confusing legacy SSR names
* There is another important clue in pLG02: names such as Sat_122 versus Satt122, and Sat_141b versus Satt141, are visually similar but represent different marker labels. We should explicitly establish their genotype relationships rather than assume anything based on nomenclature.

In [ ]:
# Cell 03.32
# Diagnose similarly named legacy markers in pLG02.
#
# These marker names must remain distinct.

name_conflict_pairs = [
    ("Sat_122",  "Satt122"),
    ("Sat_141b", "Satt141"),
    ("Satt309a", "Satt309b"),
]


name_pair_rows = []


for a, b in name_conflict_pairs:

    if (
        a in geno_92.columns
        and
        b in geno_92.columns
    ):

        stats = pairwise_linkage_stats(
            geno_92,
            a,
            b
        )

        missing_a = (
            geno_92[a]
            .isna()
            .mean()
        )

        missing_b = (
            geno_92[b]
            .isna()
            .mean()
        )


        name_pair_rows.append({
            "marker_a": a,
            "marker_b": b,
            **stats,
            "missing_a": missing_a,
            "missing_b": missing_b
        })


name_conflict_diagnostics = pd.DataFrame(
    name_pair_rows
)


print("LEGACY SSR NAME-CONFLICT DIAGNOSTICS")
print("=" * 120)

display(
    name_conflict_diagnostics
)

### Cell 03.33 — create the structurally corrected framework map

In [ ]:
# Cell 03.33
# Implement the evidence-supported pLG01 split WITHOUT overwriting
# the original frozen linkage-map checkpoint.
#
# pLG01 positions 1-22 -> pLG01a_Gm06
# pLG01 positions 23-28 -> pLG01b_Gm20
#
# Genetic coordinates are reset to zero within each new fragment.

framework_map_structural_v2 = (
    framework_map.copy()
)

framework_map_structural_v2[
    "source_working_group"
] = (
    framework_map_structural_v2[
        "working_group"
    ]
)


framework_map_structural_v2[
    "structural_group"
] = (
    framework_map_structural_v2[
        "working_group"
    ]
)


# Gm06 portion
mask_gm06 = (
    (
        framework_map_structural_v2[
            "working_group"
        ] == "pLG01"
    )
    &
    (
        framework_map_structural_v2[
            "order_position"
        ] <= 22
    )
)

framework_map_structural_v2.loc[
    mask_gm06,
    "structural_group"
] = "pLG01a_Gm06"


# Gm20 portion
mask_gm20 = (
    (
        framework_map_structural_v2[
            "working_group"
        ] == "pLG01"
    )
    &
    (
        framework_map_structural_v2[
            "order_position"
        ] >= 23
    )
)

framework_map_structural_v2.loc[
    mask_gm20,
    "structural_group"
] = "pLG01b_Gm20"


# Recalculate order position within structural groups.
framework_map_structural_v2[
    "structural_order_position"
] = (
    framework_map_structural_v2
    .groupby(
        "structural_group"
    )
    .cumcount()
    + 1
)


# Reset cumulative genetic distance to zero
# within each structural group.
framework_map_structural_v2[
    "kosambi_cm_structural_provisional"
] = (
    framework_map_structural_v2[
        "kosambi_cm_provisional"
    ]
    -
    framework_map_structural_v2
    .groupby(
        "structural_group"
    )[
        "kosambi_cm_provisional"
    ]
    .transform("min")
)


print("STRUCTURALLY CORRECTED FRAMEWORK")
print("=" * 120)

print(
    "Original linkage groups:",
    framework_map[
        "working_group"
    ].nunique()
)

print(
    "Structural groups after pLG01 split:",
    framework_map_structural_v2[
        "structural_group"
    ].nunique()
)


display(
    framework_map_structural_v2.loc[
        framework_map_structural_v2[
            "source_working_group"
        ] == "pLG01",
        [
            "source_working_group",
            "structural_group",
            "structural_order_position",
            "marker",
            "kosambi_cm_provisional",
            "kosambi_cm_structural_provisional"
        ]
    ]
)

### Cell 03.34 — remove the artificial pLG01 joining interval

In [ ]:
# Cell 03.34
# Construct a structural interval table in which the unsupported
# PUBTONC2 -> Satt367 joining interval is removed.
#
# All other frozen interval statistics remain untouched.

intervals_structural_v2 = (
    intervals_frozen.copy()
)


# Remove only the evidence-supported false-join edge.
cut_mask = (
    (
        intervals_structural_v2[
            "working_group"
        ] == "pLG01"
    )
    &
    (
        intervals_structural_v2[
            "marker_a"
        ] == "PUBTONC2"
    )
    &
    (
        intervals_structural_v2[
            "marker_b"
        ] == "Satt367"
    )
)


removed_interval = (
    intervals_structural_v2.loc[
        cut_mask
    ]
    .copy()
)


intervals_structural_v2 = (
    intervals_structural_v2.loc[
        ~cut_mask
    ]
    .copy()
)


print("REMOVED STRUCTURAL JOIN")
print("=" * 120)

display(
    removed_interval
)


original_total = (
    intervals_frozen[
        "kosambi_cm_provisional"
    ].sum()
)

structural_total = (
    intervals_structural_v2[
        "kosambi_cm_provisional"
    ].sum()
)


print(
    "\nOriginal provisional Kosambi total:",
    round(original_total, 3),
    "cM"
)

print(
    "After removing unsupported pLG01 join:",
    round(structural_total, 3),
    "cM"
)

print(
    "Removed distance:",
    round(
        original_total
        -
        structural_total,
        3
    ),
    "cM"
)

### Cell 03.35 — identify which pLG02 markers create the 21 cross-block links

In [ ]:
# Cell 03.35
# Identify the specific markers responsible for connectivity
# between the early and late portions of pLG02.

plg02_early_late[
    "core_link"
] = (
    (plg02_early_late["n_overlap"] >= 40)
    &
    (plg02_early_late["R_ril"] <= 0.30)
    &
    (plg02_early_late["lod"] >= 3.5)
)


plg02_cross_core = (
    plg02_early_late.loc[
        plg02_early_late[
            "core_link"
        ]
    ]
    .copy()
)


print("pLG02 CROSS-BLOCK CORE LINKS")
print("=" * 120)

display(
    plg02_cross_core
    .sort_values(
        [
            "R_ril",
            "lod"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# Count how many cross-block core edges each early marker has.
early_bridge_summary = (
    plg02_cross_core
    .groupby(
        "marker_a"
    )
    .agg(
        n_cross_core_links=(
            "marker_b",
            "nunique"
        ),
        min_R=(
            "R_ril",
            "min"
        ),
        median_R=(
            "R_ril",
            "median"
        ),
        max_LOD=(
            "lod",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        [
            "n_cross_core_links",
            "median_R"
        ],
        ascending=[
            False,
            True
        ]
    )
)


# Same from the late side.
late_bridge_summary = (
    plg02_cross_core
    .groupby(
        "marker_b"
    )
    .agg(
        n_cross_core_links=(
            "marker_a",
            "nunique"
        ),
        min_R=(
            "R_ril",
            "min"
        ),
        median_R=(
            "R_ril",
            "median"
        ),
        max_LOD=(
            "lod",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        [
            "n_cross_core_links",
            "median_R"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print("\nEARLY-SIDE BRIDGE MARKERS")
print("-" * 120)

display(
    early_bridge_summary
)


print("\nLATE-SIDE BRIDGE MARKERS")
print("-" * 120)

display(
    late_bridge_summary
)

### Cell 03.36 — scan every possible pLG02 cut for graph bottlenecks
* Rather than assuming position 11|12 is the relevant division, this checks every possible cut.

In [ ]:
# Cell 03.36
# Search all possible pLG02 cuts for a bottleneck in the
# original core-linkage graph.
#
# This is a GRAPH diagnostic, not a new map-order optimization.

plg02_markers_ordered = (
    plg02_ordered[
        "marker"
    ].tolist()
)


# Calculate every pair once.
all_pair_rows = []

for i, marker_a in enumerate(
    plg02_markers_ordered
):

    for j in range(
        i + 1,
        len(plg02_markers_ordered)
    ):

        marker_b = (
            plg02_markers_ordered[j]
        )

        stats = pairwise_linkage_stats(
            geno_92,
            marker_a,
            marker_b
        )

        all_pair_rows.append({
            "position_a": i + 1,
            "position_b": j + 1,
            "marker_a": marker_a,
            "marker_b": marker_b,
            **stats
        })


plg02_all_pairs = pd.DataFrame(
    all_pair_rows
)


plg02_all_pairs[
    "core_link"
] = (
    (plg02_all_pairs["n_overlap"] >= 40)
    &
    (plg02_all_pairs["R_ril"] <= 0.30)
    &
    (plg02_all_pairs["lod"] >= 3.5)
)


cut_rows = []


# Avoid trivial terminal cuts:
# require at least 3 markers on each side.
for cut_after in range(
    3,
    len(plg02_markers_ordered) - 2
):

    crossing = (
        plg02_all_pairs.loc[
            (
                plg02_all_pairs[
                    "position_a"
                ] <= cut_after
            )
            &
            (
                plg02_all_pairs[
                    "position_b"
                ] > cut_after
            )
        ]
    )


    crossing_core = (
        crossing.loc[
            crossing[
                "core_link"
            ]
        ]
    )


    cut_rows.append({
        "cut_after_position":
            cut_after,

        "left_marker":
            plg02_markers_ordered[
                cut_after - 1
            ],

        "right_marker":
            plg02_markers_ordered[
                cut_after
            ],

        "n_crossing_pairs":
            len(crossing),

        "n_crossing_core_links":
            len(crossing_core),

        "core_link_fraction":
            (
                len(crossing_core)
                /
                len(crossing)
            ),

        "best_cross_R":
            crossing[
                "R_ril"
            ].min(),

        "best_cross_LOD":
            crossing[
                "lod"
            ].max()
    })


plg02_cut_scan = pd.DataFrame(
    cut_rows
)


print("pLG02 GRAPH BOTTLENECK SCAN")
print("=" * 120)


display(
    plg02_cut_scan
    .sort_values(
        [
            "n_crossing_core_links",
            "core_link_fraction"
        ]
    )
    .head(15)
)

### Cell 03.37 — inspect the pLG02 terminal bottleneck

In [ ]:
# Cell 03.37
# Inspect the pLG02 terminal region around the strongest graph bottleneck.
#
# Candidate terminal block:
# positions 30-33 = Satt324, OEO21000, Satt303b, Satt398b

plg02_terminal = (
    plg02_ordered.loc[
        plg02_ordered["order_position"] >= 30,
        "marker"
    ]
    .tolist()
)

plg02_main = (
    plg02_ordered.loc[
        plg02_ordered["order_position"] <= 29,
        "marker"
    ]
    .tolist()
)

print("pLG02 MAIN BLOCK")
print(plg02_main)

print("\npLG02 TERMINAL BLOCK")
print(plg02_terminal)


terminal_cross_rows = []

for a in plg02_main:
    for b in plg02_terminal:

        stats = pairwise_linkage_stats(
            geno_92,
            a,
            b
        )

        terminal_cross_rows.append({
            "main_marker": a,
            "terminal_marker": b,
            **stats
        })


plg02_terminal_cross = pd.DataFrame(
    terminal_cross_rows
)

plg02_terminal_cross["core_link"] = (
    (plg02_terminal_cross["n_overlap"] >= 40)
    &
    (plg02_terminal_cross["R_ril"] <= 0.30)
    &
    (plg02_terminal_cross["lod"] >= 3.5)
)


print("\nSTRONGEST MAIN-vs-TERMINAL LINKS")
print("=" * 120)

display(
    plg02_terminal_cross
    .sort_values(
        ["R_ril", "lod"],
        ascending=[True, False]
    )
    .head(30)
)


print(
    "\nCore links:",
    int(plg02_terminal_cross["core_link"].sum()),
    "of",
    len(plg02_terminal_cross)
)

### Cell 03.38 — compare pLG02 main, terminal, and cross-block cohesion

In [ ]:
# Cell 03.38
# Compare internal cohesion of the pLG02 terminal block
# with its linkage to the main body of the group.

plg02_main_internal = block_pairwise_summary(
    geno_92,
    plg02_main,
    "main_within"
)

plg02_terminal_internal = block_pairwise_summary(
    geno_92,
    plg02_terminal,
    "terminal_within"
)


cross_for_summary = (
    plg02_terminal_cross
    .rename(
        columns={
            "main_marker": "marker_a",
            "terminal_marker": "marker_b"
        }
    )
    .copy()
)

cross_for_summary["comparison"] = "main_vs_terminal"


plg02_terminal_structure = pd.concat(
    [
        plg02_main_internal,
        plg02_terminal_internal,
        cross_for_summary[
            plg02_main_internal.columns
        ]
    ],
    ignore_index=True
)


plg02_terminal_structure["core_link"] = (
    (plg02_terminal_structure["n_overlap"] >= 40)
    &
    (plg02_terminal_structure["R_ril"] <= 0.30)
    &
    (plg02_terminal_structure["lod"] >= 3.5)
)


plg02_terminal_summary = (
    plg02_terminal_structure
    .groupby("comparison")
    .agg(
        n_pairs=("R_ril", "size"),
        median_R=("R_ril", "median"),
        min_R=("R_ril", "min"),
        max_R=("R_ril", "max"),
        median_LOD=("lod", "median"),
        max_LOD=("lod", "max"),
        n_core_links=("core_link", "sum")
    )
    .reset_index()
)


plg02_terminal_summary["core_link_fraction"] = (
    plg02_terminal_summary["n_core_links"]
    /
    plg02_terminal_summary["n_pairs"]
)


print("pLG02 TERMINAL-BLOCK COHESION TEST")
print("=" * 120)

display(
    plg02_terminal_summary
)

### Cell 03.39 — fix the exact-vs-assay-family priority bug
* This is important because our earlier deduplication could allow an assay_family observation to replace an available exact observation for the same base assay.

In [ ]:
# Cell 03.39
# Correct exact-vs-assay-family priority.
#
# Exact marker evidence must take precedence over suffix-derived
# assay-family evidence for the same:
# working_group + base_assay + physical chromosome.

combined_anchor_evidence_corrected = (
    combined_anchor_evidence.copy()
)


# Standardize chromosome column name for downstream analysis.
combined_anchor_evidence_corrected = (
    combined_anchor_evidence_corrected.rename(
        columns={
            "physical_chromosome": "physical_chr"
        }
    )
)


# Exact evidence gets priority over assay-family evidence.
evidence_priority = {
    "exact": 0,
    "assay_family": 1
}


combined_anchor_evidence_corrected[
    "evidence_priority"
] = (
    combined_anchor_evidence_corrected[
        "evidence_class"
    ]
    .map(evidence_priority)
    .fillna(99)
)


# Keep one independent observation per:
# group + base assay + chromosome,
# preferring exact evidence.
independent_assay_evidence_corrected = (
    combined_anchor_evidence_corrected
    .sort_values(
        [
            "working_group",
            "base_assay",
            "physical_chr",
            "evidence_priority"
        ]
    )
    .drop_duplicates(
        subset=[
            "working_group",
            "base_assay",
            "physical_chr"
        ],
        keep="first"
    )
    .copy()
)


print("CORRECTED INDEPENDENT ASSAY EVIDENCE")
print("=" * 110)

print(
    "Raw evidence rows:",
    len(combined_anchor_evidence_corrected)
)

print(
    "Independent group-assay-chromosome observations:",
    len(independent_assay_evidence_corrected)
)


priority_check = (
    independent_assay_evidence_corrected
    .groupby(
        [
            "working_group",
            "physical_chr",
            "evidence_class"
        ]
    )
    .size()
    .reset_index(
        name="n_assays"
    )
)


print("\npLG01 AND pLG02 CHECK")
print("-" * 110)

display(
    priority_check.loc[
        priority_check[
            "working_group"
        ].isin(
            ["pLG01", "pLG02"]
        )
    ]
)

### Cell 03.40 — rebuild corrected chromosome-vote summaries

In [ ]:
# Cell 03.40
# Rebuild physical chromosome vote summaries
# from the corrected exact-priority evidence table.

assay_vote_summary_corrected = (
    independent_assay_evidence_corrected
    .groupby(
        [
            "working_group",
            "physical_chr"
        ]
    )
    .agg(
        n_independent_assays=(
            "base_assay",
            "nunique"
        ),

        n_exact_assays=(
            "evidence_class",
            lambda x: (x == "exact").sum()
        ),

        n_family_assays=(
            "evidence_class",
            lambda x: (x == "assay_family").sum()
        )
    )
    .reset_index()
)


assay_vote_summary_corrected = (
    assay_vote_summary_corrected
    .sort_values(
        [
            "working_group",
            "n_independent_assays",
            "n_exact_assays"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)


print("CORRECTED PHYSICAL CHROMOSOME VOTES")
print("=" * 120)

display(
    assay_vote_summary_corrected
)


print("\nKEY GROUPS")
print("-" * 120)

display(
    assay_vote_summary_corrected.loc[
        assay_vote_summary_corrected[
            "working_group"
        ].isin(
            [
                "pLG01",
                "pLG02",
                "pLG13",
                "pLG16"
            ]
        )
    ]
)

### Cell 03.41 — document the pLG02 split evidence

In [ ]:
# Cell 03.41
# Summarize evidence supporting a structural split of pLG02
# after original position 29 (Satt130 | Satt324).

plg02_split_evidence = pd.DataFrame({
    "comparison": [
        "pLG02a within (positions 1-29)",
        "pLG02b within (positions 30-33)",
        "pLG02a vs pLG02b"
    ],
    "n_pairs": [
        406,
        6,
        116
    ],
    "median_R": [
        0.272121,
        0.269841,
        0.457627
    ],
    "median_LOD": [
        3.275939,
        3.329748,
        0.104361
    ],
    "n_core_links": [
        199,
        3,
        1
    ],
    "core_link_fraction": [
        199 / 406,
        3 / 6,
        1 / 116
    ]
})

print("pLG02 STRUCTURAL SPLIT EVIDENCE")
print("=" * 100)

display(plg02_split_evidence)

print("\nInterpretation:")
print(
    "The terminal four-marker block has internal linkage comparable "
    "to the main block but essentially no broad linkage to it."
)
print(
    "Only 1 of 116 cross-block pairs satisfies the core-link criterion."
)

### Cell 03.42 — create the structurally corrected 22-fragment framework

In [ ]:
# Cell 03.42
# Extend the structural correction:
#
# pLG01 -> pLG01a_Gm06 + pLG01b_Gm20
# pLG02 -> pLG02a + pLG02b
#
# All other groups remain unchanged.

framework_map_structural_v3 = (
    framework_map_structural_v2.copy()
)


# Identify the current pLG02 rows.
mask_plg02 = (
    framework_map_structural_v3["working_group"] == "pLG02"
)


# Split according to original order position.
framework_map_structural_v3.loc[
    mask_plg02
    & (
        framework_map_structural_v3["order_position"]
        <= 29
    ),
    "structural_group"
] = "pLG02a"


framework_map_structural_v3.loc[
    mask_plg02
    & (
        framework_map_structural_v3["order_position"]
        >= 30
    ),
    "structural_group"
] = "pLG02b"


# Reset within-fragment structural order.
framework_map_structural_v3[
    "structural_order"
] = (
    framework_map_structural_v3
    .groupby("structural_group")
    .cumcount()
    + 1
)


print("STRUCTURAL FRAMEWORK V3")
print("=" * 100)

print(
    "Framework markers:",
    len(framework_map_structural_v3)
)

print(
    "Structural fragments:",
    framework_map_structural_v3[
        "structural_group"
    ].nunique()
)


print("\npLG02 STRUCTURAL FRAGMENTS")
print("-" * 100)

display(
    framework_map_structural_v3.loc[
        framework_map_structural_v3[
            "structural_group"
        ].isin(
            ["pLG02a", "pLG02b"]
        )
    ][
        [
            "structural_group",
            "structural_order",
            "marker",
            "kosambi_cm_provisional"
        ]
    ]
)

### Cell 03.43 — quantify the map-length effect of removing the pLG02 join

In [ ]:
# Cell 03.43A
# Find the interval DataFrame currently available in memory.

candidate_names = [
    "provisional_framework_intervals",
    "framework_intervals",
    "intervals",
    "interval_table",
    "ordered_intervals",
    "map_intervals",
    "support_aware_intervals"
]

print("SEARCHING FOR INTERVAL TABLE")
print("=" * 100)

found = []

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            found.append(name)

            print(f"\nFOUND: {name}")
            print("Shape:", obj.shape)
            print("Columns:")
            print(obj.columns.tolist())


# Also search all DataFrames in memory for tables
# containing Satt130 and Satt324 somewhere.
print("\n\nSEARCHING ALL DATAFRAMES FOR Satt130 / Satt324")
print("=" * 100)

for name, obj in globals().copy().items():

    if not isinstance(obj, pd.DataFrame):
        continue

    try:
        text = obj.astype(str)

        has_satt130 = text.eq("Satt130").any().any()
        has_satt324 = text.eq("Satt324").any().any()

        if has_satt130 and has_satt324:
            print(f"\n{name}")
            print("Shape:", obj.shape)
            print("Columns:")
            print(obj.columns.tolist())

    except Exception:
        pass

In [ ]:
# Cell 03.43
# Quantify the pLG02 join removed by splitting
# between Satt130 and Satt324.

plg02_removed_interval = (
    intervals_frozen.loc[
        (
            intervals_frozen["working_group"] == "pLG02"
        )
        &
        (
            intervals_frozen["marker_a"] == "Satt130"
        )
        &
        (
            intervals_frozen["marker_b"] == "Satt324"
        )
    ]
    .copy()
)


print("REMOVED pLG02 JOIN INTERVAL")
print("=" * 100)

display(plg02_removed_interval)


removed_plg02_kosambi = (
    plg02_removed_interval[
        "kosambi_cm_provisional"
    ].sum()
)


# Use the already-corrected pLG01 structural total
previous_structural_total = 1286.814

updated_structural_total = (
    previous_structural_total
    - removed_plg02_kosambi
)


print(
    f"\nPreviously corrected total: "
    f"{previous_structural_total:.3f} cM"
)

print(
    f"Removed pLG02 join: "
    f"{removed_plg02_kosambi:.3f} cM"
)

print(
    f"Updated structural total: "
    f"{updated_structural_total:.3f} cM"
)

### Cell 03.44 — rebuild chromosome assignments from corrected evidence
* Now that the evidence-priority bug is fixed, let's regenerate the dominant physical-chromosome assignments cleanly rather than relying on the old Cell 03.15 table.

In [ ]:
# Cell 03.44
# Rebuild dominant chromosome assignments from corrected
# independent-assay evidence.

vote_ranked = (
    assay_vote_summary_corrected
    .sort_values(
        [
            "working_group",
            "n_independent_assays",
            "n_exact_assays"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .copy()
)


dominant_corrected = (
    vote_ranked
    .groupby(
        "working_group",
        as_index=False
    )
    .head(1)
    .copy()
)


second_corrected = (
    vote_ranked
    .groupby(
        "working_group",
        as_index=False
    )
    .nth(1)
    .reset_index()
)


dominant_corrected = dominant_corrected.rename(
    columns={
        "physical_chr": "dominant_chr",
        "n_independent_assays": "dominant_n",
        "n_exact_assays": "dominant_exact_n",
        "n_family_assays": "dominant_family_n"
    }
)


second_corrected = second_corrected[
    [
        "working_group",
        "physical_chr",
        "n_independent_assays"
    ]
].rename(
    columns={
        "physical_chr": "second_chr",
        "n_independent_assays": "second_n"
    }
)


physical_assignments_corrected = (
    dominant_corrected[
        [
            "working_group",
            "dominant_chr",
            "dominant_n",
            "dominant_exact_n",
            "dominant_family_n"
        ]
    ]
    .merge(
        second_corrected,
        on="working_group",
        how="left"
    )
)


physical_assignments_corrected[
    "dominant_fraction_vs_top2"
] = (
    physical_assignments_corrected[
        "dominant_n"
    ]
    /
    (
        physical_assignments_corrected[
            "dominant_n"
        ]
        +
        physical_assignments_corrected[
            "second_n"
        ].fillna(0)
    )
)


print("CORRECTED DOMINANT PHYSICAL-CHROMOSOME ASSIGNMENTS")
print("=" * 110)

display(
    physical_assignments_corrected
    .sort_values("working_group")
)

### Cell 03.45 — transfer physical-anchor evidence to the 22 structural fragments

In [ ]:
# Cell 03.45
# Transfer corrected independent SSR physical evidence
# from the original working groups onto the 22 structural fragments.

# Marker -> structural group lookup
marker_to_structural_group = (
    framework_map_structural_v3[
        [
            "working_group",
            "marker",
            "structural_group",
            "structural_order"
        ]
    ]
    .drop_duplicates(
        subset=["working_group", "marker"]
    )
    .copy()
)


# Attach structural-group identity to corrected
# independent physical-anchor observations.
structural_anchor_evidence = (
    independent_assay_evidence_corrected
    .merge(
        marker_to_structural_group,
        on=["working_group", "marker"],
        how="left",
        validate="many_to_one"
    )
    .copy()
)


print("STRUCTURAL PHYSICAL-ANCHOR EVIDENCE")
print("=" * 110)

print(
    "Independent anchor observations:",
    len(structural_anchor_evidence)
)

print(
    "Assigned to structural fragment:",
    structural_anchor_evidence[
        "structural_group"
    ].notna().sum()
)

print(
    "Missing structural assignment:",
    structural_anchor_evidence[
        "structural_group"
    ].isna().sum()
)


# Any missing rows should be inspected.
if structural_anchor_evidence["structural_group"].isna().any():

    print("\nUNMATCHED ANCHOR ROWS")
    print("-" * 110)

    display(
        structural_anchor_evidence.loc[
            structural_anchor_evidence[
                "structural_group"
            ].isna()
        ][
            [
                "working_group",
                "marker",
                "base_assay",
                "physical_chr",
                "evidence_class"
            ]
        ]
    )

### Cell 03.46 — summarize chromosome votes for all 22 structural fragments

In [ ]:
# Cell 03.46
# Summarize independent physical chromosome evidence
# separately for each structural linkage-group fragment.

structural_vote_summary = (
    structural_anchor_evidence
    .groupby(
        [
            "structural_group",
            "physical_chr"
        ]
    )
    .agg(
        n_independent_assays=(
            "base_assay",
            "nunique"
        ),

        n_exact_assays=(
            "evidence_class",
            lambda x: (x == "exact").sum()
        ),

        n_family_assays=(
            "evidence_class",
            lambda x: (x == "assay_family").sum()
        )
    )
    .reset_index()
)


structural_vote_summary = (
    structural_vote_summary
    .sort_values(
        [
            "structural_group",
            "n_independent_assays",
            "n_exact_assays"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .copy()
)


print("PHYSICAL CHROMOSOME VOTES BY STRUCTURAL FRAGMENT")
print("=" * 110)

display(structural_vote_summary)

### Cell 03.47 — inspect the four corrected fragments specifically

In [ ]:
# Cell 03.47
# Inspect physical evidence for the two pLG01 fragments
# and the two newly separated pLG02 fragments.

key_structural_groups = [
    "pLG01a_Gm06",
    "pLG01b_Gm20",
    "pLG02a",
    "pLG02b"
]


print("PHYSICAL EVIDENCE FOR STRUCTURALLY CORRECTED GROUPS")
print("=" * 120)


for group in key_structural_groups:

    print(f"\n{group}")
    print("-" * 120)

    group_votes = (
        structural_vote_summary.loc[
            structural_vote_summary[
                "structural_group"
            ] == group
        ]
    )

    display(group_votes)


    print("Anchor markers:")

    display(
        structural_anchor_evidence.loc[
            structural_anchor_evidence[
                "structural_group"
            ] == group
        ][
            [
                "structural_order",
                "marker",
                "base_assay",
                "physical_chr",
                "physical_bp",
                "evidence_class"
            ]
        ]
        .sort_values(
            [
                "structural_order",
                "physical_chr"
            ]
        )
    )

### Cell 03.48 — generate dominant chromosome assignments for the 22 fragments

In [ ]:
# Cell 03.48
# Generate a conservative dominant chromosome summary
# for each structural linkage-group fragment.

structural_vote_ranked = (
    structural_vote_summary
    .sort_values(
        [
            "structural_group",
            "n_independent_assays",
            "n_exact_assays"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .copy()
)


structural_dominant = (
    structural_vote_ranked
    .groupby(
        "structural_group",
        as_index=False
    )
    .head(1)
    .copy()
)


structural_second = (
    structural_vote_ranked
    .groupby(
        "structural_group",
        as_index=False
    )
    .nth(1)
    .reset_index()
)


structural_dominant = (
    structural_dominant
    .rename(
        columns={
            "physical_chr": "dominant_chr",
            "n_independent_assays": "dominant_n",
            "n_exact_assays": "dominant_exact_n",
            "n_family_assays": "dominant_family_n"
        }
    )
)


structural_second = (
    structural_second[
        [
            "structural_group",
            "physical_chr",
            "n_independent_assays"
        ]
    ]
    .rename(
        columns={
            "physical_chr": "second_chr",
            "n_independent_assays": "second_n"
        }
    )
)


structural_physical_assignments = (
    structural_dominant[
        [
            "structural_group",
            "dominant_chr",
            "dominant_n",
            "dominant_exact_n",
            "dominant_family_n"
        ]
    ]
    .merge(
        structural_second,
        on="structural_group",
        how="left"
    )
)


structural_physical_assignments[
    "dominant_fraction_top2"
] = (
    structural_physical_assignments[
        "dominant_n"
    ]
    /
    (
        structural_physical_assignments[
            "dominant_n"
        ]
        +
        structural_physical_assignments[
            "second_n"
        ].fillna(0)
    )
)


print("DOMINANT PHYSICAL CHROMOSOME BY STRUCTURAL FRAGMENT")
print("=" * 120)

display(
    structural_physical_assignments
    .sort_values("structural_group")
)

### Cell 03.49 — reset cM coordinates within all 22 structural fragments
* Your current pLG02b still begins at 191.18 cM because it inherited the old pLG02 cumulative coordinate. 
* We need each structural fragment to start at 0.

In [ ]:
# Cell 03.49
# Recalculate provisional structural cM coordinates so that
# every structural fragment begins at 0 cM.

framework_map_structural_v3 = (
    framework_map_structural_v3
    .sort_values(
        [
            "structural_group",
            "structural_order"
        ]
    )
    .copy()
)


framework_map_structural_v3[
    "kosambi_cm_structural_provisional"
] = (
    framework_map_structural_v3[
        "kosambi_cm_provisional"
    ]
    -
    framework_map_structural_v3
    .groupby("structural_group")[
        "kosambi_cm_provisional"
    ]
    .transform("min")
)


framework_map_structural_v3[
    "haldane_cm_structural_provisional"
] = (
    framework_map_structural_v3[
        "haldane_cm_provisional"
    ]
    -
    framework_map_structural_v3
    .groupby("structural_group")[
        "haldane_cm_provisional"
    ]
    .transform("min")
)


print("STRUCTURAL COORDINATES RESET")
print("=" * 110)

print(
    "Markers:",
    len(framework_map_structural_v3)
)

print(
    "Structural fragments:",
    framework_map_structural_v3[
        "structural_group"
    ].nunique()
)


# Verify every fragment starts at zero.
start_check = (
    framework_map_structural_v3
    .groupby("structural_group")
    .agg(
        n_markers=("marker", "size"),
        start_cm=(
            "kosambi_cm_structural_provisional",
            "min"
        ),
        end_cm=(
            "kosambi_cm_structural_provisional",
            "max"
        )
    )
    .reset_index()
)


display(start_check)

assert (
    start_check["start_cm"].abs() < 1e-10
).all()

print("\nAll 22 structural fragments start at 0 cM.")

### Cell 03.50 — construct the final 144-interval structural table
* Rather than manually deleting joins, this derives retained intervals from whether the two adjacent markers belong to the same structural fragment.

In [ ]:
# Cell 03.50
# Build structural interval table automatically.
#
# An original interval is retained only when both markers
# remain in the same structural fragment.

structural_lookup = (
    framework_map_structural_v3[
        [
            "working_group",
            "marker",
            "structural_group"
        ]
    ]
    .drop_duplicates()
)


left_lookup = (
    structural_lookup
    .rename(
        columns={
            "marker": "marker_a",
            "structural_group": "structural_group_a"
        }
    )
)


right_lookup = (
    structural_lookup
    .rename(
        columns={
            "marker": "marker_b",
            "structural_group": "structural_group_b"
        }
    )
)


intervals_structural_v3_all = (
    intervals_frozen
    .merge(
        left_lookup,
        on=["working_group", "marker_a"],
        how="left",
        validate="many_to_one"
    )
    .merge(
        right_lookup,
        on=["working_group", "marker_b"],
        how="left",
        validate="many_to_one"
    )
)


# Retain only intervals whose two markers remain
# in the same structural fragment.
intervals_structural_v3 = (
    intervals_structural_v3_all.loc[
        intervals_structural_v3_all[
            "structural_group_a"
        ]
        ==
        intervals_structural_v3_all[
            "structural_group_b"
        ]
    ]
    .copy()
)


intervals_structural_v3[
    "structural_group"
] = (
    intervals_structural_v3[
        "structural_group_a"
    ]
)


removed_structural_joins = (
    intervals_structural_v3_all.loc[
        intervals_structural_v3_all[
            "structural_group_a"
        ]
        !=
        intervals_structural_v3_all[
            "structural_group_b"
        ]
    ]
    .copy()
)


print("FINAL STRUCTURAL INTERVAL TABLE")
print("=" * 110)

print(
    "Original intervals:",
    len(intervals_frozen)
)

print(
    "Retained structural intervals:",
    len(intervals_structural_v3)
)

print(
    "Removed structural joins:",
    len(removed_structural_joins)
)


print("\nREMOVED JOINS")
display(
    removed_structural_joins[
        [
            "working_group",
            "marker_a",
            "marker_b",
            "R_ril",
            "lod",
            "kosambi_cm_provisional",
            "structural_group_a",
            "structural_group_b"
        ]
    ]
)


final_structural_kosambi = (
    intervals_structural_v3[
        "kosambi_cm_provisional"
    ].sum()
)


print(
    f"\nFinal provisional structural map length: "
    f"{final_structural_kosambi:.3f} cM"
)

### Cell 03.51 — conservative physical-assignment confidence

In [ ]:
# Cell 03.51
# Assign conservative confidence categories to physical
# chromosome assignments of the 22 structural fragments.

structural_physical_assignments_final = (
    structural_physical_assignments.copy()
)


def classify_physical_assignment(row):

    n = row["dominant_n"]
    exact = row["dominant_exact_n"]
    family = row["dominant_family_n"]
    frac = row["dominant_fraction_top2"]

    second_n = (
        0
        if pd.isna(row["second_n"])
        else row["second_n"]
    )

    # Strong multi-anchor assignment
    if (
        n >= 5
        and exact >= 3
        and frac >= 0.75
    ):
        return "strong"

    # Supported by several mostly exact independent assays
    if (
        n >= 3
        and exact >= 2
        and frac >= 0.75
    ):
        return "supported"

    # Exact + family evidence, but limited or mixed
    if (
        n >= 3
        and exact >= 1
        and second_n == 0
    ):
        return "supported_exact_plus_family"

    # Two exact anchors without competing evidence
    if (
        n >= 2
        and exact >= 2
        and second_n == 0
    ):
        return "tentative_exact"

    # Family-only chromosome evidence
    if (
        exact == 0
        and family >= 2
    ):
        return "tentative_family_only"

    # Multiple anchors, but competing chromosome evidence
    if (
        n >= 2
        and second_n > 0
    ):
        return "tentative_mixed"

    # Only one informative anchor
    if n == 1:
        return "insufficient_single_anchor"

    return "unresolved"


structural_physical_assignments_final[
    "assignment_status"
] = (
    structural_physical_assignments_final
    .apply(
        classify_physical_assignment,
        axis=1
    )
)


print("FINAL CONSERVATIVE PHYSICAL ASSIGNMENTS")
print("=" * 120)

display(
    structural_physical_assignments_final[
        [
            "structural_group",
            "dominant_chr",
            "dominant_n",
            "dominant_exact_n",
            "dominant_family_n",
            "second_chr",
            "second_n",
            "dominant_fraction_top2",
            "assignment_status"
        ]
    ]
    .sort_values("structural_group")
)

### Cell 03.52 — final structural-map summary before export

In [ ]:
# Cell 03.52
# Generate final structural framework summary.
# No files are overwritten yet.

structural_group_summary_final = (
    framework_map_structural_v3
    .groupby("structural_group")
    .agg(
        n_markers=("marker", "size"),

        kosambi_length_cm=(
            "kosambi_cm_structural_provisional",
            "max"
        ),

        haldane_length_cm=(
            "haldane_cm_structural_provisional",
            "max"
        )
    )
    .reset_index()
)


structural_group_summary_final = (
    structural_group_summary_final
    .merge(
        structural_physical_assignments_final[
            [
                "structural_group",
                "dominant_chr",
                "dominant_n",
                "dominant_exact_n",
                "dominant_family_n",
                "second_chr",
                "second_n",
                "assignment_status"
            ]
        ],
        on="structural_group",
        how="left"
    )
)


print("FINAL STRUCTURAL FRAMEWORK SUMMARY")
print("=" * 120)

display(structural_group_summary_final)


print("\nGLOBAL SUMMARY")
print("-" * 120)

print(
    "Ordered framework markers:",
    len(framework_map_structural_v3)
)

print(
    "Structural linkage-group fragments:",
    structural_group_summary_final[
        "structural_group"
    ].nunique()
)

print(
    "Retained intervals:",
    len(intervals_structural_v3)
)

print(
    "Associated unordered satellites:",
    16
)

print(
    "Total group-associated markers:",
    182
)

print(
    f"Total provisional Kosambi length: "
    f"{intervals_structural_v3['kosambi_cm_provisional'].sum():.3f} cM"
)


print("\nPHYSICAL ASSIGNMENT STATUS COUNTS")
print("-" * 120)

display(
    structural_physical_assignments_final[
        "assignment_status"
    ]
    .value_counts()
    .rename_axis("assignment_status")
    .reset_index(name="n_fragments")
)

### Cell 03.53 — attach physical assignment metadata to the 166-marker structural map

In [ ]:
# Cell 03.53
# Attach conservative physical chromosome assignment metadata
# to every marker in the 166-marker structural framework.

framework_map_structural_final = (
    framework_map_structural_v3
    .merge(
        structural_physical_assignments_final[
            [
                "structural_group",
                "dominant_chr",
                "dominant_n",
                "dominant_exact_n",
                "dominant_family_n",
                "second_chr",
                "second_n",
                "dominant_fraction_top2",
                "assignment_status"
            ]
        ],
        on="structural_group",
        how="left",
        validate="many_to_one"
    )
    .copy()
)


print("FINAL STRUCTURAL FRAMEWORK WITH PHYSICAL ASSIGNMENTS")
print("=" * 120)

print("Markers:", len(framework_map_structural_final))

print(
    "Structural fragments:",
    framework_map_structural_final[
        "structural_group"
    ].nunique()
)

print(
    "Markers in fragments with a physical chromosome candidate:",
    framework_map_structural_final[
        "dominant_chr"
    ].notna().sum()
)


display(
    framework_map_structural_final[
        [
            "structural_group",
            "structural_order",
            "marker",
            "kosambi_cm_structural_provisional",
            "dominant_chr",
            "assignment_status"
        ]
    ]
    .head(30)
)

### Cell 03.54 — create a final map-validation summary

In [ ]:
# Cell 03.54
# Final integrity checks before export.

final_map_validation = pd.DataFrame({
    "metric": [
        "ordered_framework_markers",
        "structural_fragments",
        "retained_intervals",
        "removed_structural_joins",
        "unordered_satellites",
        "total_group_associated_markers",
        "provisional_kosambi_cm"
    ],

    "value": [
        len(framework_map_structural_final),

        framework_map_structural_final[
            "structural_group"
        ].nunique(),

        len(intervals_structural_v3),

        len(removed_structural_joins),

        16,

        182,

        intervals_structural_v3[
            "kosambi_cm_provisional"
        ].sum()
    ]
})


print("FINAL MAP VALIDATION")
print("=" * 100)

display(final_map_validation)


# Hard checks
assert len(framework_map_structural_final) == 166

assert (
    framework_map_structural_final[
        "structural_group"
    ].nunique()
    == 22
)

assert len(intervals_structural_v3) == 144

assert len(removed_structural_joins) == 2

assert abs(
    intervals_structural_v3[
        "kosambi_cm_provisional"
    ].sum()
    - 1272.119
) < 0.01


print("\nALL FINAL STRUCTURAL-MAP CHECKS PASSED.")

### Cell 03.55 — export the frozen structural + physical map checkpoint

In [ ]:
# Cell 03.55 — corrected
# Export final frozen structural + physical map checkpoint.
#
# If satellite_final_status is not present in notebook memory,
# reload the frozen unordered-satellite table from the earlier
# linkage-map checkpoint.

from pathlib import Path
import pandas as pd


project_root = PROJECT_ROOT


# ------------------------------------------------------------------
# 1. Recover unordered satellites if not already in notebook memory
# ------------------------------------------------------------------

if "satellite_final_status" not in globals():

    previous_checkpoint = (
        project_root
        / "results"
        / "linkage_map"
        / "flyer_hartwig_provisional_framework_map.xlsx"
    )

    print(
        "satellite_final_status not in memory.\n"
        "Reloading unordered satellites from frozen checkpoint..."
    )

    satellite_final_status = pd.read_excel(
        previous_checkpoint,
        sheet_name="unordered_satellites"
    )


print(
    "Unordered satellites loaded:",
    len(satellite_final_status)
)

assert len(satellite_final_status) == 16, (
    "Expected 16 unordered satellites, "
    f"but found {len(satellite_final_status)}."
)


# ------------------------------------------------------------------
# 2. Define new structural/physical checkpoint
# ------------------------------------------------------------------

output_file = (
    project_root
    / "results"
    / "linkage_map"
    / "flyer_hartwig_structural_physical_map_final.xlsx"
)


# ------------------------------------------------------------------
# 3. Export
# ------------------------------------------------------------------

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    framework_map_structural_final.to_excel(
        writer,
        sheet_name="ordered_framework",
        index=False
    )

    intervals_structural_v3.to_excel(
        writer,
        sheet_name="retained_intervals",
        index=False
    )

    removed_structural_joins.to_excel(
        writer,
        sheet_name="removed_joins",
        index=False
    )

    structural_group_summary_final.to_excel(
        writer,
        sheet_name="group_summary",
        index=False
    )

    structural_physical_assignments_final.to_excel(
        writer,
        sheet_name="physical_assignments",
        index=False
    )

    structural_vote_summary.to_excel(
        writer,
        sheet_name="physical_votes",
        index=False
    )

    structural_anchor_evidence.to_excel(
        writer,
        sheet_name="anchor_evidence",
        index=False
    )

    satellite_final_status.to_excel(
        writer,
        sheet_name="unordered_satellites",
        index=False
    )

    final_map_validation.to_excel(
        writer,
        sheet_name="validation_summary",
        index=False
    )


# ------------------------------------------------------------------
# 4. Verify the workbook after writing
# ------------------------------------------------------------------

print("\nFINAL STRUCTURAL/PHYSICAL MAP EXPORTED")
print("=" * 110)

print(output_file)
print("\nFile exists:", output_file.exists())

if output_file.exists():

    print(
        "File size:",
        round(
            output_file.stat().st_size / 1024,
            2
        ),
        "KB"
    )

    exported_sheets = pd.ExcelFile(
        output_file
    ).sheet_names

    print("\nExported sheets:")
    for sheet in exported_sheets:
        print("  -", sheet)

    assert len(exported_sheets) == 9

    print(
        "\nFINAL STRUCTURAL/PHYSICAL "
        "CHECKPOINT VERIFIED."
    )

### Cell 03.56 — first chromosome-coverage overview
* Now we change gears from map reconstruction to coverage assessment.

In [ ]:
# Cell 03.56
# Summarize how the 22 structural fragments are distributed
# across candidate physical soybean chromosomes.

chromosome_fragment_summary = (
    structural_physical_assignments_final
    .groupby("dominant_chr")
    .agg(
        n_structural_fragments=(
            "structural_group",
            "nunique"
        ),

        total_dominant_anchors=(
            "dominant_n",
            "sum"
        ),

        total_exact_anchors=(
            "dominant_exact_n",
            "sum"
        ),

        structural_fragments=(
            "structural_group",
            lambda x: ", ".join(sorted(x))
        )
    )
    .reset_index()
    .sort_values("dominant_chr")
)


print("STRUCTURAL FRAGMENTS BY CANDIDATE PHYSICAL CHROMOSOME")
print("=" * 120)

display(chromosome_fragment_summary)


observed_chr = set(
    chromosome_fragment_summary[
        "dominant_chr"
    ].dropna()
)


expected_chr = {
    f"Gm{i:02d}"
    for i in range(1, 21)
}


missing_chr = sorted(
    expected_chr - observed_chr
)


print(
    "\nPhysical chromosomes represented by at least "
    "one candidate structural fragment:",
    len(observed_chr)
)

print(
    "Chromosomes without a current dominant assignment:",
    missing_chr
)

### Cell 03.57 — chromosome-level framework coverage
* This summarizes how many framework markers and how much genetic-map length are currently associated with each dominant chromosome.

In [ ]:
# Cell 03.57 — corrected
# Summarize structural framework coverage by candidate physical chromosome.
#
# structural_group_summary_final ALREADY contains the physical-assignment
# columns, so no additional merge is needed.

print("AVAILABLE COLUMNS")
print("=" * 100)
print(structural_group_summary_final.columns.tolist())


chromosome_framework_coverage = (
    structural_group_summary_final
    .groupby(
        "dominant_chr",
        dropna=False
    )
    .agg(
        n_fragments=(
            "structural_group",
            "nunique"
        ),

        n_framework_markers=(
            "n_markers",
            "sum"
        ),

        total_kosambi_cm=(
            "kosambi_length_cm",
            "sum"
        ),

        fragments=(
            "structural_group",
            lambda x: ", ".join(sorted(x))
        )
    )
    .reset_index()
    .sort_values(
        "dominant_chr",
        na_position="last"
    )
)


print("\nFRAMEWORK COVERAGE BY DOMINANT PHYSICAL CHROMOSOME")
print("=" * 120)

display(chromosome_framework_coverage)


print(
    "\nTotal framework markers represented:",
    chromosome_framework_coverage[
        "n_framework_markers"
    ].sum()
)

print(
    "Total provisional map length represented:",
    round(
        chromosome_framework_coverage[
            "total_kosambi_cm"
        ].sum(),
        3
    ),
    "cM"
)


# Integrity checks
assert (
    chromosome_framework_coverage[
        "n_framework_markers"
    ].sum()
    == 166
)

assert abs(
    chromosome_framework_coverage[
        "total_kosambi_cm"
    ].sum()
    - 1272.119396
) < 0.01


print("\nCOVERAGE SUMMARY CHECKS PASSED.")

### Cell 03.58 — distinguish absent dominant assignments from any physical evidence
* This is important because “no dominant chromosome assignment” is not the same as “no evidence anywhere.”

In [ ]:
# Cell 03.58
# Determine which soybean chromosomes occur anywhere in the
# independent SSR evidence, even if they are not dominant assignments.

expected_chr = [
    f"Gm{i:02d}"
    for i in range(1, 21)
]


dominant_chr_set = set(
    structural_physical_assignments_final[
        "dominant_chr"
    ].dropna()
)


any_anchor_chr_set = set(
    structural_anchor_evidence[
        "physical_chr"
    ].dropna()
)


chromosome_presence_audit = pd.DataFrame({
    "physical_chr": expected_chr
})


chromosome_presence_audit[
    "has_dominant_fragment"
] = (
    chromosome_presence_audit[
        "physical_chr"
    ].isin(dominant_chr_set)
)


chromosome_presence_audit[
    "appears_anywhere_in_anchor_evidence"
] = (
    chromosome_presence_audit[
        "physical_chr"
    ].isin(any_anchor_chr_set)
)


anchor_counts = (
    structural_anchor_evidence
    .groupby("physical_chr")
    .agg(
        total_anchor_observations=(
            "base_assay",
            "nunique"
        ),

        exact_anchor_observations=(
            "evidence_class",
            lambda x: (x == "exact").sum()
        ),

        family_anchor_observations=(
            "evidence_class",
            lambda x: (x == "assay_family").sum()
        )
    )
    .reset_index()
)


chromosome_presence_audit = (
    chromosome_presence_audit
    .merge(
        anchor_counts,
        on="physical_chr",
        how="left"
    )
    .fillna({
        "total_anchor_observations": 0,
        "exact_anchor_observations": 0,
        "family_anchor_observations": 0
    })
)


print("PHYSICAL-CHROMOSOME PRESENCE AUDIT")
print("=" * 120)

display(chromosome_presence_audit)


print("\nNO DOMINANT FRAGMENT:")
print(
    chromosome_presence_audit.loc[
        ~chromosome_presence_audit[
            "has_dominant_fragment"
        ],
        "physical_chr"
    ].tolist()
)


print("\nNO PHYSICAL ANCHOR EVIDENCE AT ALL:")
print(
    chromosome_presence_audit.loc[
        ~chromosome_presence_audit[
            "appears_anywhere_in_anchor_evidence"
        ],
        "physical_chr"
    ].tolist()
)

### Cell 03.59 — identify fragmented chromosomes
* Several physical chromosomes have more than one linkage fragment. We should explicitly catalog those before QTL analysis.

In [ ]:
# Cell 03.59
# Identify candidate physical chromosomes represented by
# more than one independent structural linkage fragment.

fragmented_chromosomes = (
    chromosome_framework_coverage.loc[
        chromosome_framework_coverage[
            "n_fragments"
        ] > 1
    ]
    .copy()
)


print("PHYSICAL CHROMOSOMES REPRESENTED BY MULTIPLE STRUCTURAL FRAGMENTS")
print("=" * 120)

display(fragmented_chromosomes)


print(
    "\nNumber of chromosomes represented by multiple fragments:",
    len(fragmented_chromosomes)
)

### Cell 03.60 — create the QTL-ready map table
* This makes a clean table specifically for the next notebook.

In [ ]:
# Cell 03.60
# Create a compact QTL-ready map table from the frozen
# structural framework.

qtl_ready_map = (
    framework_map_structural_final[
        [
            "structural_group",
            "structural_order",
            "marker",
            "kosambi_cm_structural_provisional",
            "dominant_chr",
            "assignment_status"
        ]
    ]
    .rename(
        columns={
            "structural_group": "linkage_fragment",
            "structural_order": "marker_order",
            "kosambi_cm_structural_provisional": "position_cm",
            "dominant_chr": "candidate_physical_chr"
        }
    )
    .sort_values(
        [
            "linkage_fragment",
            "marker_order"
        ]
    )
    .reset_index(drop=True)
)


print("QTL-READY STRUCTURAL MAP")
print("=" * 110)

print("Markers:", len(qtl_ready_map))

print(
    "Linkage fragments:",
    qtl_ready_map[
        "linkage_fragment"
    ].nunique()
)

display(qtl_ready_map.head(40))